# stt-workbench: training notebook (Kaggle & Colab)

Train a **speech-to-text** model on a free Kaggle or Colab GPU, with any of the approaches
of the `stt-workbench` repo:

| `MODEL` | what it is |
|---|---|
| `ctc` | encoder + CTC head: the fastest baseline |
| `aed` | attention encoder-decoder (Transformer decoder) |
| `las` | Listen-Attend-Spell: LSTM decoder with location-aware attention |
| `hybrid` | CTC + attention, joint loss and joint decoding (ESPnet default) |
| `rnnt` | RNN-Transducer: streaming-friendly |
| `wav2vec2` | fine-tune a pretrained wav2vec2 with a CTC head (Hugging Face) |
| `whisper` | fine-tune a pretrained Whisper (Hugging Face) |

with `ENCODER` = `bilstm`, `transformer` or `conformer` for the first five.

This notebook is **self-contained**: all the code is copied in from the repo, with the same
defaults, and its checkpoints work with the repo's `stt-transcribe`, `stt-eval` and
`stt-train --resume`.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Data | your dataset as input, or `DOWNLOAD_DATASET` | Google Drive, upload, or `DOWNLOAD_DATASET` |
| Session limit | 12 h per session, 30 h GPU / week | ~12 h, can disconnect when idle |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

**Just trying it?** Set `SMOKE_TEST = True`: a small synthetic dataset is written and every
cell runs in a few minutes on CPU or GPU.

**Sections:** 1 Install · 2 Config · 3 Data · 4 Model · 5 Decoding & metrics · 6 Training code
· 7 Train (live curves) · 8 Pretrained fine-tuning (wav2vec2 / Whisper) · 9 Results · 10 Download

## 1. Install

Kaggle and Colab already have PyTorch, torchaudio (RNN-T loss), NumPy, SciPy, matplotlib
and `transformers`. This cell adds `soundfile` and `sentencepiece`.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile>=0.12", "sentencepiece>=0.2"])

import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Config

### 2.1 Config classes (same as `src/stt/config.py`)

All hyperparameters live here, grouped by what they control.

Configs are frozen dataclasses: they cannot be changed by accident after creation, and
they are saved inside every checkpoint, so a trained model can be rebuilt without
remembering any flag (`load_checkpoint` in train.py).

The defaults are deliberately SMALL (a few million parameters at most): this repo exists
to learn how each ASR approach works, not to chase leaderboard numbers. Every size can be
changed from the command line (`stt-train --help`) or in the notebook.

Model types (`ModelConfig.model`):
    ctc     encoder + CTC head                               (models.CTCModel)
    aed     encoder + Transformer decoder with cross-attention (models.AttentionModel)
    las     encoder + LSTM decoder with location-aware attention (models.AttentionModel)
    hybrid  CTC head + Transformer decoder, joint loss and joint decoding (models.AttentionModel)
    rnnt    encoder + prediction network + joiner, transducer loss (models.TransducerModel)
Encoders (`ModelConfig.encoder`): bilstm | transformer | conformer   (encoders.py)

In [ ]:
from dataclasses import asdict, dataclass, fields
from pathlib import Path
from typing import Any

MODEL_TYPES = ("ctc", "aed", "las", "hybrid", "rnnt")
ENCODER_TYPES = ("bilstm", "transformer", "conformer")
DECODER_METHODS = ("greedy", "beam")


@dataclass(frozen=True, slots=True)
class DataConfig:
    """Where the speech is, how it is turned into features, and how it is batched.

    A dataset is a folder with `train.jsonl` and `val.jsonl` manifests (see docs/data.md):
    one JSON object per line, `{"audio": "<path>", "text": "<transcript>", "duration": <s>}`.
    Paths are absolute or relative to the manifest's folder.
    """

    data_dir: Path = Path("data/synthetic")
    train_manifest: str = "train.jsonl"
    val_manifest: str = "val.jsonl"
    sample_rate: int = 16_000  # audio is resampled to this on load
    # Log-mel front end (features.py): 25 ms windows every 10 ms is the ASR standard.
    n_mels: int = 80
    win_ms: float = 25.0
    hop_ms: float = 10.0
    f_min: float = 20.0
    f_max: float | None = None  # None = sample_rate / 2
    # Text: "char" (one token per character) or "bpe" (SentencePiece model at `bpe_model`).
    tokenizer: str = "char"
    bpe_model: Path | None = None
    lowercase: bool = True
    # SpecAugment (training only). Masks are in feature frames / mel bins.
    specaugment: bool = True
    freq_masks: int = 2
    freq_mask_width: int = 15
    time_masks: int = 2
    time_mask_width: int = 40
    time_mask_max_ratio: float = 0.2  # a time mask never hides more than this fraction of the clip
    # Batching: clips are sorted by duration into buckets so that padding is small.
    max_duration_s: float = 20.0  # longer training clips are skipped
    bucket_size: int = 64  # clips per duration bucket (batches are drawn inside a bucket)

    def __post_init__(self) -> None:
        if self.sample_rate <= 0 or self.n_mels <= 0:
            raise ValueError("sample_rate and n_mels must be positive")
        if self.tokenizer not in ("char", "bpe"):
            raise ValueError("tokenizer must be 'char' or 'bpe'")
        if self.tokenizer == "bpe" and self.bpe_model is None:
            raise ValueError("tokenizer='bpe' needs bpe_model")

    @property
    def n_fft(self) -> int:
        """Smallest power of two holding the analysis window."""
        win = self.win_length
        n = 1
        while n < win:
            n *= 2
        return n

    @property
    def win_length(self) -> int:
        return round(self.sample_rate * self.win_ms / 1000)

    @property
    def hop_length(self) -> int:
        return round(self.sample_rate * self.hop_ms / 1000)


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Shape of the recogniser: which approach, which encoder, and how big."""

    model: str = "ctc"  # one of MODEL_TYPES
    encoder: str = "conformer"  # one of ENCODER_TYPES
    # Conv2d subsampling in front of every encoder: 2 layers of stride 2 = 4x fewer frames
    # (40 ms per encoder frame). 4 ms-per-frame features are too many for attention to look at.
    subsampling_channels: int = 64
    subsampling_factor: int = 4  # 2 or 4
    d_model: int = 144  # encoder width
    n_heads: int = 4
    n_layers: int = 6
    ffn_mult: int = 4  # feed-forward width = ffn_mult * d_model
    conv_kernel: int = 15  # Conformer convolution module kernel (odd)
    dropout: float = 0.1
    # Attention decoders (aed / hybrid: Transformer decoder; las: LSTM decoder).
    dec_layers: int = 2
    dec_dim: int = 144
    label_smoothing: float = 0.1
    # Hybrid: loss = ctc_weight * CTC + (1 - ctc_weight) * attention; also used in joint decoding.
    ctc_weight: float = 0.3
    # RNN-T: prediction network (LSTM over previous tokens) and joiner.
    pred_dim: int = 128
    pred_layers: int = 1
    joint_dim: int = 128

    def __post_init__(self) -> None:
        if self.model not in MODEL_TYPES:
            raise ValueError(f"model must be one of {MODEL_TYPES}, got {self.model!r}")
        if self.encoder not in ENCODER_TYPES:
            raise ValueError(f"encoder must be one of {ENCODER_TYPES}, got {self.encoder!r}")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        if self.subsampling_factor not in (2, 4):
            raise ValueError("subsampling_factor must be 2 or 4")
        if self.conv_kernel % 2 == 0:
            raise ValueError("conv_kernel must be odd")
        if not 0.0 <= self.ctc_weight <= 1.0:
            raise ValueError("ctc_weight must be in [0, 1]")

    @property
    def has_ctc(self) -> bool:
        return self.model in ("ctc", "hybrid")

    @property
    def has_attention_decoder(self) -> bool:
        return self.model in ("aed", "las", "hybrid")


@dataclass(frozen=True, slots=True)
class TrainConfig:
    """How long, how fast, where to save, how to evaluate."""

    output_dir: Path = Path("checkpoints/run")
    batch_size: int = 16
    max_steps: int = 20_000
    max_epochs: int | None = None  # stop after this many epochs if set (max_steps still applies)
    lr: float = 1e-3
    warmup_steps: int = 1_000  # linear warm-up, then cosine decay to `min_lr`
    min_lr: float = 1e-5
    weight_decay: float = 1e-2
    grad_clip: float = 5.0
    precision: str = "fp32"  # fp32 | bf16 | fp16 (GPU autocast; CTC/RNN-T losses stay in fp32)
    num_workers: int = 0
    device: str = "auto"  # auto | cpu | cuda
    seed: int = 0
    log_every: int = 50
    eval_every: int = 500  # steps; also at the end
    eval_batches: int | None = None  # None = whole validation manifest
    decode: str = "greedy"  # validation decoding: greedy | beam
    beam_size: int = 4
    select_by: str = "wer"  # best.pt keeps the lowest of: wer | cer | loss
    time_budget_hours: float | None = None  # stop cleanly (after saving) before a session limit
    resume: bool = True  # continue from output_dir/last.pt if it exists

    def __post_init__(self) -> None:
        if self.precision not in ("fp32", "bf16", "fp16"):
            raise ValueError("precision must be fp32, bf16 or fp16")
        if self.decode not in DECODER_METHODS:
            raise ValueError(f"decode must be one of {DECODER_METHODS}")
        if self.select_by not in ("wer", "cer", "loss"):
            raise ValueError("select_by must be wer, cer or loss")


def config_to_dict(cfg: Any) -> dict[str, Any]:
    """A JSON-friendly dict (Paths as strings)."""
    out = {}
    for k, v in asdict(cfg).items():
        out[k] = str(v) if isinstance(v, Path) else v
    return out


def config_from_dict(cls: type, d: dict[str, Any]) -> Any:
    """Inverse of config_to_dict; unknown keys (older/newer checkpoints) are ignored."""
    kwargs = {}
    for f in fields(cls):
        if f.name not in d:
            continue
        v = d[f.name]
        if v is not None and ("Path" in str(f.type)) and isinstance(v, str):
            v = Path(v)
        kwargs[f.name] = v
    return cls(**kwargs)

### 2.2 Settings for this run

**Data.** A dataset folder holds `train.jsonl` and `val.jsonl` manifests (one JSON object
per line: `audio`, `text`, `duration`; see the repo's `docs/data.md`). Either:

- **Kaggle:** add a dataset as input: a folder with the two manifests, or a LibriSpeech /
  LJSpeech folder (manifests are written for it); the cell below finds it;
- **Colab:** put it on Google Drive (`USE_GOOGLE_DRIVE = True`) and set `DATA_DIR`;
- **Either:** set `DOWNLOAD_DATASET` to `"yesno"` (4.5 MB), `"fsdd"` (15 MB),
  `"librispeech-dev-clean"` (337 MB) or `"librispeech-train-clean-100"` (6 GB + dev-clean).

The batch size and precision adapt to the GPU without changing what is learned.

In [ ]:
import dataclasses
import os
from pathlib import Path

MODEL = "hybrid"            # ctc | aed | las | hybrid | rnnt | wav2vec2 | whisper
ENCODER = "conformer"       # bilstm | transformer | conformer   (not used by wav2vec2 / whisper)
SMOKE_TEST = False          # True = quick check of every cell (synthetic data, small model)
DOWNLOAD_DATASET = None     # None | "yesno" | "fsdd" | "librispeech-dev-clean" | "librispeech-train-clean-100"
USE_GOOGLE_DRIVE = False    # Colab only: data + checkpoints on Drive (survive disconnects)
DATA_DIR = None             # None = find it automatically (Kaggle input, /content/data, ./data)
SAMPLE_RATE = 16000         # 8000 for yesno / fsdd / the synthetic data (set automatically for those)
TOKENIZER = "char"          # "char" | "bpe" (a SentencePiece model is trained in section 3)
BPE_VOCAB = 300
RESUME = True               # continue from last.pt in OUTPUT_DIR if it exists
TIME_BUDGET_HOURS = 11.0    # stop cleanly (after saving) before the 12 h session limit
MAX_STEPS = 20000
PRETRAINED_NAME = {"wav2vec2": "facebook/wav2vec2-base", "whisper": "openai/whisper-base"}.get(MODEL)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(f"/content/drive/MyDrive/stt-workbench/{MODEL}-{ENCODER}")
elif ON_KAGGLE:
    OUTPUT_DIR = Path(f"/kaggle/working/checkpoints/{MODEL}-{ENCODER}")
elif ON_COLAB:
    OUTPUT_DIR = Path(f"/content/checkpoints/{MODEL}-{ENCODER}")
else:
    OUTPUT_DIR = Path(f"checkpoints/notebook-{MODEL}-{ENCODER}")
SCRATCH = Path("/tmp/stt_data")  # downloads (outside /kaggle/working: not counted in the 20 GB output)


def find_data_dir() -> Path | None:
    """First folder with train.jsonl + val.jsonl, or a LibriSpeech / LJSpeech folder, in the usual places."""
    roots = [Path("/kaggle/input"), Path("/content/data"), Path("/content/drive/MyDrive"), Path("data")]
    for root in roots:
        if not root.exists():
            continue
        candidates = [root, *sorted(p for p in root.rglob("*") if p.is_dir())][:2000]
        for path in candidates:
            if (path / "train.jsonl").exists() and (path / "val.jsonl").exists():
                return path
        for path in candidates:
            if path.name == "LibriSpeech" or (path / "metadata.csv").exists():
                return path
    return None


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    gpu_gb = props.total_memory / 1e9
    precision = "bf16" if props.major >= 8 else "fp16"
    batch_size = 64 if gpu_gb >= 30 else 32
    print(f"GPU: {props.name} ({gpu_gb:.0f} GB), compute capability {props.major}.{props.minor}")
else:
    precision, batch_size = "fp32", 8
    print("No GPU: enable one in the notebook settings (CPU is fine only for SMOKE_TEST).")

model_config = ModelConfig(model=MODEL if MODEL in MODEL_TYPES else "ctc", encoder=ENCODER)
train_config = TrainConfig(
    output_dir=OUTPUT_DIR, batch_size=batch_size, precision=precision, max_steps=MAX_STEPS, warmup_steps=1000,
    num_workers=min(4, os.cpu_count() or 1), device=device.type, log_every=50, eval_every=1000,
    time_budget_hours=TIME_BUDGET_HOURS, resume=RESUME, decode="greedy",
)
if SMOKE_TEST:
    OUTPUT_DIR = OUTPUT_DIR.parent / f"smoke-{MODEL}-{ENCODER}"
    model_config = dataclasses.replace(model_config, d_model=64, n_layers=2, dec_dim=64, dec_layers=1, ffn_mult=2)
    train_config = dataclasses.replace(
        train_config, output_dir=OUTPUT_DIR, max_steps=200, warmup_steps=20, batch_size=8, log_every=10,
        eval_every=100, num_workers=0, time_budget_hours=0.5,
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("model:", MODEL, "| encoder:", ENCODER, "| output:", OUTPUT_DIR)

## 3. Data

### 3.1 Tokenizer code (same as `src/stt/tokenizer.py`)

Turning text into integer tokens and back.

Every model needs a few special ids:
    blank (0)  CTC / RNN-T "emit nothing" symbol. Also the padding value of label tensors.
    sos   (1)  start of sentence, fed to attention decoders as the first input
    eos   (2)  end of sentence, the attention decoders learn to emit it to stop
    unk   (3)  anything not in the vocabulary
Keeping blank at 0 matches `torch.nn.CTCLoss` and `torchaudio.functional.rnnt_loss` defaults.

Two tokenizers share this interface:
    CharTokenizer      one token per character (space included). Simple, no training, the
                       right choice for small data: the model only has to learn ~30 classes.
    SentencePieceTokenizer  sub-word pieces (BPE/unigram) from a `.model` file; fewer, longer
                       tokens shorten the output sequence, which helps attention decoders and
                       RNN-T on big datasets. Needs the `bpe` extra (sentencepiece).

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

BLANK, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ["<blank>", "<sos>", "<eos>", "<unk>"]


def normalize_text(text: str, lowercase: bool = True) -> str:
    """Collapse whitespace, optionally lowercase. Keeps apostrophes and letters of any script."""
    text = text.strip()
    if lowercase:
        text = text.lower()
    return re.sub(r"\s+", " ", text)


class CharTokenizer:
    """Character vocabulary built from the training transcripts."""

    def __init__(self, chars: list[str]):
        self.itos = SPECIALS + list(chars)
        self.stoi = {c: i for i, c in enumerate(self.itos)}

    @classmethod
    def from_texts(cls, texts: list[str]) -> CharTokenizer:
        chars = sorted({c for t in texts for c in t})
        return cls(chars)

    @property
    def vocab_size(self) -> int:
        return len(self.itos)

    def encode(self, text: str) -> list[int]:
        return [self.stoi.get(c, UNK) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.itos[i] for i in ids if i >= len(SPECIALS))

    def to_dict(self) -> dict:
        return {"type": "char", "chars": self.itos[len(SPECIALS) :]}

    @classmethod
    def from_dict(cls, d: dict) -> CharTokenizer:
        return cls(d["chars"])

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.to_dict(), ensure_ascii=False, indent=1))


class SentencePieceTokenizer:
    """Sub-word tokenizer around a trained SentencePiece model (`pip install sentencepiece`).

    Ids are shifted by len(SPECIALS) so the four special ids above keep their meaning.
    Train a model with `train_sentencepiece(texts, prefix, vocab_size)`.
    """

    def __init__(self, model_path: str | Path):
        import sentencepiece as spm  # lazy: optional dependency

        self.model_path = str(model_path)
        self.sp = spm.SentencePieceProcessor(model_file=self.model_path)
        self.offset = len(SPECIALS)

    @property
    def vocab_size(self) -> int:
        return self.offset + self.sp.get_piece_size()

    def encode(self, text: str) -> list[int]:
        return [i + self.offset for i in self.sp.encode(text, out_type=int)]

    def decode(self, ids: list[int]) -> str:
        pieces = [i - self.offset for i in ids if i >= self.offset]
        return self.sp.decode(pieces)

    def to_dict(self) -> dict:
        return {"type": "bpe", "model_path": self.model_path}

    @classmethod
    def from_dict(cls, d: dict) -> SentencePieceTokenizer:
        return cls(d["model_path"])


def train_sentencepiece(texts: list[str], prefix: str | Path, vocab_size: int = 256, model_type: str = "bpe") -> Path:
    """Train a SentencePiece model on `texts`; returns the `.model` path."""
    import tempfile

    import sentencepiece as spm

    prefix = Path(prefix)
    prefix.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
        f.write("\n".join(texts))
        corpus = f.name
    spm.SentencePieceTrainer.train(
        input=corpus, model_prefix=str(prefix), vocab_size=vocab_size, model_type=model_type,
        character_coverage=1.0, bos_id=-1, eos_id=-1, unk_id=0, pad_id=-1,
    )
    return prefix.with_suffix(".model")


def tokenizer_from_dict(d: dict):
    """Rebuild whichever tokenizer was saved in a checkpoint."""
    if d["type"] == "char":
        return CharTokenizer.from_dict(d)
    if d["type"] == "bpe":
        return SentencePieceTokenizer.from_dict(d)
    raise ValueError(f"unknown tokenizer type {d['type']!r}")

### 3.2 Feature code (same as `src/stt/features.py`)

From a waveform to the log-mel spectrogram every model in this repo listens to.

The pipeline (the classic one, from Kaldi to Whisper):

1. load the file (`soundfile`) and resample to the configured rate (`scipy.signal.resample_poly`);
2. short-time Fourier transform with 25 ms Hann windows every 10 ms (`torch.stft`);
3. power spectrum |X|^2, then a bank of `n_mels` triangular filters spaced on the mel scale,
   mel(f) = 2595 log10(1 + f / 700), so that low frequencies (where vowels live) get more bins;
4. log(. + 1e-6), then per-utterance mean/variance normalisation (CMVN) so the model does
   not care about recording level;
5. SpecAugment (Park et al. 2019) during training: a few random frequency bands and time
   spans are zeroed, which makes the model rely on context instead of single cues.

Shapes: waveform (T,) -> features (frames, n_mels) with frames = T // hop + 1.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly



def load_audio(path: str | Path, sample_rate: int) -> np.ndarray:
    """Mono float32 waveform at `sample_rate`, in [-1, 1]."""
    wav, sr = sf.read(str(path), dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != sample_rate:
        wav = resample(wav, sr, sample_rate)
    return np.ascontiguousarray(wav, dtype=np.float32)


def resample(wav: np.ndarray, orig_sr: int, new_sr: int) -> np.ndarray:
    """Polyphase resampling (anti-aliased), exact for integer ratios."""
    if orig_sr == new_sr:
        return wav
    g = math.gcd(orig_sr, new_sr)
    return resample_poly(wav, new_sr // g, orig_sr // g).astype(np.float32)


def hz_to_mel(f: float) -> float:
    return 2595.0 * math.log10(1.0 + f / 700.0)


def mel_to_hz(m: float) -> float:
    return 700.0 * (10.0 ** (m / 2595.0) - 1.0)


def mel_filterbank(sample_rate: int, n_fft: int, n_mels: int, f_min: float, f_max: float | None) -> torch.Tensor:
    """(n_fft // 2 + 1, n_mels) triangular filters, Slaney-style area normalised."""
    f_max = f_max or sample_rate / 2
    n_bins = n_fft // 2 + 1
    fft_freqs = torch.linspace(0, sample_rate / 2, n_bins)
    mel_points = torch.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2)
    hz_points = torch.tensor([mel_to_hz(m) for m in mel_points.tolist()])
    lower, center, upper = hz_points[:-2], hz_points[1:-1], hz_points[2:]
    f = fft_freqs[:, None]
    rising = (f - lower) / (center - lower).clamp(min=1e-6)
    falling = (upper - f) / (upper - center).clamp(min=1e-6)
    fb = torch.clamp(torch.minimum(rising, falling), min=0.0)
    fb = fb * (2.0 / (upper - lower))[None, :]  # equal-area filters
    return fb


class LogMel(torch.nn.Module):
    """Log-mel spectrogram front end. Works on (T,) or (B, T) waveforms."""

    def __init__(self, cfg: DataConfig):
        super().__init__()
        self.cfg = cfg
        self.n_fft = cfg.n_fft
        self.hop = cfg.hop_length
        self.win = cfg.win_length
        self.register_buffer("window", torch.hann_window(self.win), persistent=False)
        self.register_buffer(
            "fb", mel_filterbank(cfg.sample_rate, self.n_fft, cfg.n_mels, cfg.f_min, cfg.f_max), persistent=False
        )

    @torch.no_grad()
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        """(T,) -> (frames, n_mels) or (B, T) -> (B, frames, n_mels)."""
        squeeze = wav.dim() == 1
        if squeeze:
            wav = wav[None]
        spec = torch.stft(
            wav, self.n_fft, hop_length=self.hop, win_length=self.win, window=self.window,
            center=True, pad_mode="reflect", return_complex=True,
        )
        power = spec.abs().pow(2)  # (B, bins, frames)
        mel = torch.matmul(power.transpose(1, 2), self.fb)  # (B, frames, n_mels)
        logmel = torch.log(mel + 1e-6)
        return logmel[0] if squeeze else logmel

    def n_frames(self, n_samples: int) -> int:
        return n_samples // self.hop + 1


def cmvn(feats: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """Per-utterance mean/variance normalisation over time. (frames, n_mels)."""
    mean = feats.mean(dim=0, keepdim=True)
    std = feats.std(dim=0, keepdim=True)
    return (feats - mean) / (std + eps)


def spec_augment(
    feats: torch.Tensor,
    freq_masks: int = 2,
    freq_mask_width: int = 15,
    time_masks: int = 2,
    time_mask_width: int = 40,
    time_mask_max_ratio: float = 0.2,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    """SpecAugment masking on one (frames, n_mels) utterance (time warping is left out: it
    costs a lot and helps little, see the SpecAugment paper's ablation)."""
    feats = feats.clone()
    n_frames, n_mels = feats.shape
    fill = 0.0  # features are CMVN'd: zero = the mean

    def rand_int(high: int) -> int:
        if high <= 0:
            return 0
        return int(torch.randint(0, high + 1, (1,), generator=generator))

    for _ in range(freq_masks):
        w = rand_int(min(freq_mask_width, n_mels))
        f0 = rand_int(n_mels - w)
        feats[:, f0 : f0 + w] = fill
    max_t = min(time_mask_width, int(n_frames * time_mask_max_ratio))
    for _ in range(time_masks):
        w = rand_int(max_t)
        t0 = rand_int(n_frames - w)
        feats[t0 : t0 + w, :] = fill
    return feats

### 3.3 Manifest / dataset code (same as `src/stt/data.py`)

Manifests, the dataset, and batching.

A dataset is a folder with `train.jsonl` and `val.jsonl`: one JSON object per line,

    {"audio": "wavs/0001.wav", "text": "yes no yes", "duration": 2.1}

Relative audio paths are resolved against the manifest's folder. Any format soundfile
reads (wav, flac, ogg) at any sample rate works; audio is resampled on load.

`SpeechDataset` loads one clip, computes its log-mel features (features.py), applies CMVN
and (training) SpecAugment, and tokenizes the text. `collate` pads a list of items into
tensors + lengths. `BucketSampler` groups clips of similar duration into batches so that
little of the batch is padding, which matters for the sequence losses and for speed.

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset, Sampler



@dataclass
class Utterance:
    audio: Path
    text: str
    duration: float
    id: str = ""


def read_manifest(path: str | Path, lowercase: bool = True) -> list[Utterance]:
    path = Path(path)
    utts = []
    for i, line in enumerate(path.read_text().splitlines()):
        if not line.strip():
            continue
        d = json.loads(line)
        audio = Path(d["audio"])
        if not audio.is_absolute():
            audio = path.parent / audio
        utts.append(Utterance(audio, normalize_text(d["text"], lowercase), float(d.get("duration", 0.0)),
                              d.get("id", audio.stem)))
    return utts


def write_manifest(path: str | Path, utts: list[Utterance] | list[dict]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for u in utts:
            d = u if isinstance(u, dict) else {"audio": str(u.audio), "text": u.text, "duration": u.duration, "id": u.id}
            f.write(json.dumps(d, ensure_ascii=False) + "\n")


def build_tokenizer(cfg: DataConfig, utts: list[Utterance]):
    if cfg.tokenizer == "bpe":
        return SentencePieceTokenizer(cfg.bpe_model)
    return CharTokenizer.from_texts([u.text for u in utts])


class SpeechDataset(Dataset):
    def __init__(self, cfg: DataConfig, utts: list[Utterance], tokenizer, train: bool, seed: int = 0):
        self.cfg = cfg
        self.utts = [u for u in utts if not (train and u.duration > cfg.max_duration_s)]
        self.tokenizer = tokenizer
        self.train = train
        self.logmel = LogMel(cfg)
        self.generator = torch.Generator().manual_seed(seed)

    def __len__(self) -> int:
        return len(self.utts)

    def features(self, wav) -> torch.Tensor:
        feats = cmvn(self.logmel(torch.as_tensor(wav)))
        if self.train and self.cfg.specaugment:
            feats = spec_augment(feats, self.cfg.freq_masks, self.cfg.freq_mask_width, self.cfg.time_masks,
                                 self.cfg.time_mask_width, self.cfg.time_mask_max_ratio, self.generator)
        return feats

    def __getitem__(self, i: int) -> dict:
        u = self.utts[i]
        wav = load_audio(u.audio, self.cfg.sample_rate)
        return {"feats": self.features(wav), "tokens": torch.tensor(self.tokenizer.encode(u.text), dtype=torch.long),
                "text": u.text, "id": u.id}


def collate(items: list[dict]) -> dict:
    feat_lengths = torch.tensor([it["feats"].size(0) for it in items])
    target_lengths = torch.tensor([it["tokens"].numel() for it in items])
    feats = torch.zeros(len(items), int(feat_lengths.max()), items[0]["feats"].size(1))
    targets = torch.full((len(items), max(int(target_lengths.max()), 1)), BLANK, dtype=torch.long)
    for b, it in enumerate(items):
        feats[b, : it["feats"].size(0)] = it["feats"]
        targets[b, : it["tokens"].numel()] = it["tokens"]
    return {"feats": feats, "feat_lengths": feat_lengths, "targets": targets, "target_lengths": target_lengths,
            "texts": [it["text"] for it in items], "ids": [it["id"] for it in items]}


class BucketSampler(Sampler[list[int]]):
    """Batches of similar-duration clips. Clips are sorted by duration, cut into buckets of
    `bucket_size`, each bucket shuffled and cut into batches; the batch order is shuffled too."""

    def __init__(self, durations: list[float], batch_size: int, bucket_size: int, shuffle: bool, seed: int = 0):
        self.order = sorted(range(len(durations)), key=lambda i: durations[i])
        self.batch_size = batch_size
        self.bucket_size = max(bucket_size, batch_size)
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        batches = []
        for start in range(0, len(self.order), self.bucket_size):
            bucket = self.order[start : start + self.bucket_size]
            if self.shuffle:
                rng.shuffle(bucket)
            batches += [bucket[i : i + self.batch_size] for i in range(0, len(bucket), self.batch_size)]
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

    def __len__(self) -> int:
        return sum(-(-min(self.bucket_size, len(self.order) - s) // self.batch_size)
                   for s in range(0, len(self.order), self.bucket_size))


def make_loader(dataset: SpeechDataset, batch_size: int, shuffle: bool, num_workers: int = 0, seed: int = 0) -> DataLoader:
    sampler = BucketSampler([u.duration for u in dataset.utts], batch_size, dataset.cfg.bucket_size, shuffle, seed)
    return DataLoader(dataset, batch_sampler=sampler, collate_fn=collate, num_workers=num_workers)

### 3.4 Built-in dataset preparation code (same as `src/stt/datasets.py`)

Built-in datasets: `stt-data <name> --out data/<name>` writes audio (if needed) and manifests.

    synthetic    a deterministic toy spoken language, generated here, no download. Each letter
                 of a 12-letter alphabet is a short pattern of tones (a 2-formant "vowel" or a
                 noise burst + tone "consonant"); words are random letter strings; sentences
                 are 1-5 words with 200 ms pauses, played at random speed and level under
                 background noise at 8 kHz. It is learnable in minutes on a CPU, so it proves
                 that a model and a decoder work before any real data is downloaded.
    yesno        60 recordings of "yes"/"no" sequences (Hebrew, 8 kHz, openslr 1); 4.5 MB.
    fsdd         Free Spoken Digit Dataset: 3000 clips of a spoken digit, 6 speakers, 8 kHz;
                 ~15 MB. Split by speaker (held-out speakers in val).
    librispeech  openslr 12 tarballs (dev-clean 337 MB ... train-clean-100 6 GB) with
                 `*.trans.txt`; for Kaggle/Colab.
    ljspeech     LJ Speech (metadata.csv + wavs/, 2.6 GB), single speaker, 13 100 clips.
    commonvoice  a Common Voice `train.tsv`/`dev.tsv` + clips/*.mp3 folder (needs ffmpeg).

Manifests are `train.jsonl` / `val.jsonl` (data.py). Nothing is downloaded to anywhere but
`--out` (or `--download-dir`).

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import io
import random
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import soundfile as sf


YESNO_URL = "https://www.openslr.org/resources/1/waves_yesno.tar.gz"
FSDD_URL = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/heads/master.zip"
LIBRISPEECH_URL = "https://www.openslr.org/resources/12/{subset}.tar.gz"
LJSPEECH_URL = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
DIGITS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]


# ------------------------------------------------------------------------------- synthetic
SYNTH_ALPHABET = "abdeiklmnost"
SYNTH_SR = 8000


def _letter_params(letter: str) -> dict:
    """Each letter is a fixed acoustic pattern. Vowels: two formants; consonants: noise + tone."""
    i = SYNTH_ALPHABET.index(letter)
    vowel = letter in "aeio"
    rng = np.random.RandomState(1000 + i)
    return {
        "vowel": vowel,
        "f1": 250 + 90 * i + rng.uniform(0, 40),
        "f2": 900 + 170 * i + rng.uniform(0, 60),
        "chirp": rng.uniform(-300, 300),  # consonants glide
        "duration": 0.09 if vowel else 0.06,
    }


def synth_letter(letter: str, rng: np.random.RandomState, speed: float) -> np.ndarray:
    p = _letter_params(letter)
    n = int(SYNTH_SR * p["duration"] / speed)
    t = np.arange(n) / SYNTH_SR
    env = np.sin(np.pi * np.arange(n) / n) ** 0.5
    if p["vowel"]:
        f0 = 110 * rng.uniform(0.85, 1.15)
        harmonics = sum(np.sin(2 * np.pi * f0 * h * t) / h for h in range(1, 12))
        wave = harmonics * (0.5 * np.sin(2 * np.pi * p["f1"] * t) + 0.5 * np.sin(2 * np.pi * p["f2"] * t)) * 0.5
        wave = wave + 0.3 * np.sin(2 * np.pi * p["f1"] * t) + 0.2 * np.sin(2 * np.pi * p["f2"] * t)
    else:
        phase = 2 * np.pi * (p["f2"] * t + 0.5 * p["chirp"] * t**2)
        wave = 0.5 * np.sin(phase) + 0.4 * rng.randn(n) * np.exp(-t * 40)
    return (wave * env).astype(np.float32)


def synth_sentence(text: str, rng: np.random.RandomState) -> np.ndarray:
    speed = rng.uniform(0.8, 1.25)
    pieces = [np.zeros(int(SYNTH_SR * rng.uniform(0.05, 0.15)), np.float32)]
    for ch in text:
        if ch == " ":
            pieces.append(np.zeros(int(SYNTH_SR * 0.2 / speed), np.float32))
        else:
            pieces.append(synth_letter(ch, rng, speed))
            pieces.append(np.zeros(int(SYNTH_SR * 0.02), np.float32))
    pieces.append(np.zeros(int(SYNTH_SR * rng.uniform(0.05, 0.15)), np.float32))
    wav = np.concatenate(pieces)
    wav = wav * rng.uniform(0.3, 0.9) / (np.abs(wav).max() + 1e-6)
    noise = rng.randn(len(wav)).astype(np.float32) * rng.uniform(0.01, 0.06)  # "low quality"
    return np.clip(wav + noise, -1, 1).astype(np.float32)


def random_sentence(rng: np.random.RandomState, max_words: int = 5) -> str:
    words = []
    for _ in range(rng.randint(1, max_words + 1)):
        words.append("".join(rng.choice(list(SYNTH_ALPHABET), size=rng.randint(2, 6))))
    return " ".join(words)


def write_synthetic(out: Path, n_train: int = 400, n_val: int = 60, seed: int = 0, max_words: int = 5) -> Path:
    """Write wavs + manifests; deterministic for a given seed."""
    out.mkdir(parents=True, exist_ok=True)
    rng = np.random.RandomState(seed)
    for split, n in (("train", n_train), ("val", n_val)):
        (out / split).mkdir(exist_ok=True)
        rows = []
        for i in range(n):
            text = random_sentence(rng, max_words)
            wav = synth_sentence(text, rng)
            path = out / split / f"{split}_{i:05d}.wav"
            sf.write(path, wav, SYNTH_SR)
            rows.append({"audio": f"{split}/{path.name}", "text": text, "duration": len(wav) / SYNTH_SR, "id": path.stem})
        write_manifest(out / f"{split}.jsonl", rows)
    return out


# ------------------------------------------------------------------------------- downloads
def download(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return dest
    print(f"downloading {url} -> {dest}")
    tmp = dest.with_suffix(dest.suffix + ".part")
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    return dest


def wav_duration(path: Path) -> float:
    info = sf.info(str(path))
    return info.frames / info.samplerate


def stable_split(key: str, val_fraction: float) -> str:
    """Deterministic train/val assignment from a hash of the id."""
    h = int(hashlib.md5(key.encode()).hexdigest(), 16) % 1000
    return "val" if h < val_fraction * 1000 else "train"


def prepare_yesno(out: Path, download_dir: Path | None = None, val_fraction: float = 0.2) -> Path:
    archive = download(YESNO_URL, (download_dir or out) / "waves_yesno.tar.gz")
    with tarfile.open(archive) as tar:
        tar.extractall(out, filter="data")
    wavs = sorted((out / "waves_yesno").glob("*.wav"))
    rows = {"train": [], "val": []}
    for w in wavs:
        text = " ".join("yes" if c == "1" else "no" for c in w.stem.split("_"))
        rows[stable_split(w.stem, val_fraction)].append(
            {"audio": str(w.relative_to(out)), "text": text, "duration": wav_duration(w), "id": w.stem})
    for split, split_rows in rows.items():
        write_manifest(out / f"{split}.jsonl", split_rows)
    return out


def prepare_fsdd(out: Path, download_dir: Path | None = None, val_speakers: tuple[str, ...] = ("yweweler",),
                 clips_per_utterance: int = 1, seed: int = 0) -> Path:
    """FSDD. `clips_per_utterance` > 1 concatenates several digits of one speaker into one clip
    (with short pauses) so that the task becomes a sequence task, not a classification."""
    archive = download(FSDD_URL, (download_dir or out) / "fsdd.zip")
    with zipfile.ZipFile(archive) as z:
        members = [m for m in z.namelist() if m.endswith(".wav") and "/recordings/" in m]
        for m in members:
            target = out / "recordings" / Path(m).name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(m))
    wavs = sorted((out / "recordings").glob("*.wav"))
    by_speaker: dict[str, list[Path]] = {}
    for w in wavs:
        _digit, speaker, _ = w.stem.split("_", 2)
        by_speaker.setdefault(speaker, []).append(w)
    rows = {"train": [], "val": []}
    rng = random.Random(seed)
    for speaker, files in by_speaker.items():
        split = "val" if speaker in val_speakers else "train"
        rng.shuffle(files)
        if clips_per_utterance <= 1:
            for w in files:
                rows[split].append({"audio": str(w.relative_to(out)), "text": DIGITS[int(w.stem.split("_")[0])],
                                    "duration": wav_duration(w), "id": w.stem})
            continue
        (out / "sequences").mkdir(exist_ok=True)
        for i in range(0, len(files) - clips_per_utterance + 1, clips_per_utterance):
            group = files[i : i + clips_per_utterance]
            pieces, sr = [], None
            for w in group:
                wav, sr = sf.read(str(w), dtype="float32")
                pieces += [wav, np.zeros(int(sr * rng.uniform(0.1, 0.3)), np.float32)]
            wav = np.concatenate(pieces)
            path = out / "sequences" / f"{speaker}_{i // clips_per_utterance:04d}.wav"
            sf.write(path, wav, sr)
            text = " ".join(DIGITS[int(w.stem.split("_")[0])] for w in group)
            rows[split].append({"audio": str(path.relative_to(out)), "text": text, "duration": len(wav) / sr, "id": path.stem})
    for split, split_rows in rows.items():
        write_manifest(out / f"{split}.jsonl", split_rows)
    return out


def prepare_librispeech(out: Path, train_subset: str = "dev-clean", val_subset: str = "dev-clean",
                        download_dir: Path | None = None, root: Path | None = None, val_fraction: float = 0.1) -> Path:
    """`root` = an existing LibriSpeech/ folder (e.g. a Kaggle dataset); otherwise tarballs are fetched.
    If train and val subsets are the same, speakers are split (val_fraction of them held out)."""
    out.mkdir(parents=True, exist_ok=True)
    root = root or (download_dir or out) / "LibriSpeech"
    for subset in {train_subset, val_subset}:
        if (root / subset).exists():
            continue
        archive = download(LIBRISPEECH_URL.format(subset=subset), (download_dir or out) / f"{subset}.tar.gz")
        with tarfile.open(archive) as tar:
            tar.extractall(root.parent, filter="data")

    def rows_of(subset: str) -> list[dict]:
        rows = []
        for trans in sorted((root / subset).rglob("*.trans.txt")):
            for line in trans.read_text().splitlines():
                uid, text = line.split(" ", 1)
                audio = trans.parent / f"{uid}.flac"
                rows.append({"audio": str(audio), "text": text.lower(), "duration": wav_duration(audio),
                             "id": uid, "speaker": uid.split("-")[0]})
        return rows

    if train_subset == val_subset:
        rows = rows_of(train_subset)
        write_manifest(out / "train.jsonl", [r for r in rows if stable_split(r["speaker"], val_fraction) == "train"])
        write_manifest(out / "val.jsonl", [r for r in rows if stable_split(r["speaker"], val_fraction) == "val"])
    else:
        write_manifest(out / "train.jsonl", rows_of(train_subset))
        write_manifest(out / "val.jsonl", rows_of(val_subset))
    return out


def prepare_ljspeech(out: Path, root: Path | None = None, download_dir: Path | None = None, val_fraction: float = 0.05) -> Path:
    out.mkdir(parents=True, exist_ok=True)
    root = root or (download_dir or out) / "LJSpeech-1.1"
    if not root.exists():
        archive = download(LJSPEECH_URL, (download_dir or out) / "LJSpeech-1.1.tar.bz2")
        with tarfile.open(archive) as tar:
            tar.extractall(root.parent, filter="data")
    rows = {"train": [], "val": []}
    for line in (root / "metadata.csv").read_text(encoding="utf-8").splitlines():
        uid, _, normalized = line.split("|")
        audio = root / "wavs" / f"{uid}.wav"
        rows[stable_split(uid, val_fraction)].append(
            {"audio": str(audio), "text": normalized, "duration": wav_duration(audio), "id": uid})
    for split, split_rows in rows.items():
        write_manifest(out / f"{split}.jsonl", split_rows)
    return out


def prepare_commonvoice(out: Path, root: Path, train_tsv: str = "train.tsv", val_tsv: str = "dev.tsv",
                        sample_rate: int = 16_000, max_clips: int | None = None) -> Path:
    """Converts clips/*.mp3 to wav with ffmpeg (must be installed) and writes manifests."""
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg is needed to decode Common Voice mp3 clips")
    out.mkdir(parents=True, exist_ok=True)
    (out / "wavs").mkdir(exist_ok=True)
    for split, tsv in (("train", train_tsv), ("val", val_tsv)):
        lines = (root / tsv).read_text(encoding="utf-8").splitlines()
        header = lines[0].split("\t")
        i_path, i_text = header.index("path"), header.index("sentence")
        rows = []
        for line in lines[1 : (max_clips + 1 if max_clips else None)]:
            cols = line.split("\t")
            mp3 = root / "clips" / cols[i_path]
            wav = out / "wavs" / (Path(cols[i_path]).stem + ".wav")
            if not wav.exists():
                subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(mp3), "-ac", "1", "-ar",
                                str(sample_rate), str(wav)], check=True)
            rows.append({"audio": str(wav.relative_to(out)), "text": cols[i_text], "duration": wav_duration(wav),
                         "id": wav.stem})
        write_manifest(out / f"{split}.jsonl", rows)
    return out


def read_wav_bytes(data: bytes) -> tuple[np.ndarray, int]:
    wav, sr = sf.read(io.BytesIO(data), dtype="float32")
    return wav, sr


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Prepare a dataset: audio + train.jsonl / val.jsonl")
    p.add_argument("name", choices=["synthetic", "yesno", "fsdd", "librispeech", "ljspeech", "commonvoice"])
    p.add_argument("--out", type=Path, default=None, help="default data/<name>")
    p.add_argument("--download-dir", type=Path, default=None, help="where archives are kept (default: --out)")
    p.add_argument("--root", type=Path, default=None, help="existing dataset folder (librispeech/ljspeech/commonvoice)")
    p.add_argument("--n-train", type=int, default=400, help="synthetic: training sentences")
    p.add_argument("--n-val", type=int, default=60, help="synthetic: validation sentences")
    p.add_argument("--max-words", type=int, default=5, help="synthetic: words per sentence")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--clips-per-utterance", type=int, default=1, help="fsdd: digits per clip")
    p.add_argument("--train-subset", default="dev-clean", help="librispeech")
    p.add_argument("--val-subset", default="dev-clean", help="librispeech")
    p.add_argument("--max-clips", type=int, default=None, help="commonvoice: limit per split")
    args = p.parse_args(argv)
    out = args.out or Path("data") / args.name
    if args.name == "synthetic":
        write_synthetic(out, args.n_train, args.n_val, args.seed, args.max_words)
    elif args.name == "yesno":
        prepare_yesno(out, args.download_dir)
    elif args.name == "fsdd":
        prepare_fsdd(out, args.download_dir, clips_per_utterance=args.clips_per_utterance, seed=args.seed)
    elif args.name == "librispeech":
        prepare_librispeech(out, args.train_subset, args.val_subset, args.download_dir, args.root)
    elif args.name == "ljspeech":
        prepare_ljspeech(out, args.root, args.download_dir)
    elif args.name == "commonvoice":
        if args.root is None:
            raise SystemExit("--root <CommonVoice folder with train.tsv and clips/> is required")
        prepare_commonvoice(out, args.root, max_clips=args.max_clips)
    for split in ("train", "val"):
        n = sum(1 for _ in (out / f"{split}.jsonl").open())
        print(f"{split}: {n} utterances -> {out / f'{split}.jsonl'}")

### 3.5 Prepare the data

Writes the manifests (and downloads if asked), then builds the `DataConfig` and shows a few
utterances and one log-mel spectrogram.

In [ ]:
import json

if SMOKE_TEST:
    DATA_DIR = write_synthetic(Path("synthetic_data"), n_train=200, n_val=40, seed=0)
    SAMPLE_RATE = SYNTH_SR
elif DOWNLOAD_DATASET == "yesno":
    DATA_DIR, SAMPLE_RATE = prepare_yesno(SCRATCH / "yesno"), 8000
elif DOWNLOAD_DATASET == "fsdd":
    DATA_DIR, SAMPLE_RATE = prepare_fsdd(SCRATCH / "fsdd", clips_per_utterance=3), 8000
elif DOWNLOAD_DATASET and DOWNLOAD_DATASET.startswith("librispeech-"):
    subset = DOWNLOAD_DATASET.split("-", 1)[1]
    DATA_DIR = prepare_librispeech(SCRATCH / "librispeech", train_subset=subset, val_subset="dev-clean",
                                   download_dir=SCRATCH)
elif DATA_DIR is None:
    DATA_DIR = find_data_dir()
    if DATA_DIR is None:
        raise FileNotFoundError("No data found: add a dataset as input, set DATA_DIR, DOWNLOAD_DATASET, or SMOKE_TEST = True.")
DATA_DIR = Path(DATA_DIR)
if not (DATA_DIR / "train.jsonl").exists():  # a raw LibriSpeech / LJSpeech folder: write manifests next to the outputs
    manifests = OUTPUT_DIR.parent / f"manifests-{DATA_DIR.name}"
    if DATA_DIR.name == "LibriSpeech" or any(DATA_DIR.glob("*/*.trans.txt")) or any(DATA_DIR.rglob("*.trans.txt")):
        subsets = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
        train_subset = next((s for s in subsets if s.startswith("train")), subsets[0])
        val_subset = next((s for s in subsets if s.startswith("dev")), train_subset)
        DATA_DIR = prepare_librispeech(manifests, train_subset, val_subset, root=DATA_DIR)
    elif (DATA_DIR / "metadata.csv").exists():
        DATA_DIR = prepare_ljspeech(manifests, root=DATA_DIR)
print("data:", DATA_DIR)

train_utts = read_manifest(DATA_DIR / "train.jsonl")
val_utts = read_manifest(DATA_DIR / "val.jsonl")
print(f"{len(train_utts)} train / {len(val_utts)} val utterances, "
      f"{sum(u.duration for u in train_utts) / 3600:.2f} h of training audio")
for u in train_utts[:3]:
    print(f"  {u.duration:5.2f}s  {u.text!r}")

bpe_model = None
if TOKENIZER == "bpe":
    bpe_model = train_sentencepiece([u.text for u in train_utts], OUTPUT_DIR / "bpe", vocab_size=BPE_VOCAB)
data_config = DataConfig(data_dir=DATA_DIR, sample_rate=SAMPLE_RATE, tokenizer=TOKENIZER, bpe_model=bpe_model)
tokenizer = build_tokenizer(data_config, train_utts)
print("vocabulary:", tokenizer.vocab_size, "tokens")

import matplotlib.pyplot as plt

feats = cmvn(LogMel(data_config)(torch.as_tensor(load_audio(train_utts[0].audio, SAMPLE_RATE))))
plt.figure(figsize=(10, 3))
plt.imshow(feats.T, aspect="auto", origin="lower")
plt.title(train_utts[0].text)
plt.xlabel("frame (10 ms)")
plt.ylabel("mel bin")
plt.show()

## 4. Model

### 4.1 Encoders (same as `src/stt/encoders.py`)

The acoustic encoders: log-mel frames in, one context vector per (subsampled) frame out.

All three encoders share the same front end and the same output contract, so every
approach (CTC, attention, RNN-T) can use any of them:

    Conv2dSubsampling   two 3x3 stride-2 convolutions over the (time, mel) image. Each halves
                        the frame rate: 10 ms frames become 40 ms, 4x fewer positions for the
                        encoder (and for attention: quadratic cost) to deal with.
    BiLSTMEncoder       stacked bidirectional LSTMs (Graves et al. 2013, Deep Speech 2). The
                        forward pass sees the past, the backward pass the future.
    TransformerEncoder  pre-norm self-attention blocks with sinusoidal positions
                        (Vaswani et al. 2017; Speech-Transformer, Dong et al. 2018).
    ConformerEncoder    Conformer blocks (Gulati et al. 2020): macaron feed-forward halves,
                        relative-position multi-head self-attention, and a depthwise-conv
                        module for local patterns. The strongest of the three per parameter.

Every encoder's forward is `(feats, lengths) -> (out, out_lengths)` with
feats (B, T, n_mels), out (B, T', d_model); `out_lengths` are needed because batches are
padded and the losses must know where each utterance really ends.

In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn



def lengths_to_mask(lengths: torch.Tensor, max_len: int | None = None) -> torch.Tensor:
    """(B,) lengths -> (B, T) boolean mask, True on valid positions."""
    max_len = max_len or int(lengths.max())
    return torch.arange(max_len, device=lengths.device)[None, :] < lengths[:, None]


class Conv2dSubsampling(nn.Module):
    """(B, T, n_mels) -> (B, T // factor, d_model), factor 2 or 4."""

    def __init__(self, n_mels: int, d_model: int, channels: int, factor: int):
        super().__init__()
        n_convs = {2: 1, 4: 2}[factor]
        layers: list[nn.Module] = []
        c_in = 1
        freq = n_mels
        for _ in range(n_convs):
            layers += [nn.Conv2d(c_in, channels, 3, stride=2, padding=1), nn.ReLU()]
            c_in = channels
            freq = (freq + 2 - 3) // 2 + 1
        self.conv = nn.Sequential(*layers)
        self.out = nn.Linear(channels * freq, d_model)
        self.factor = factor
        self.n_convs = n_convs

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.conv(x[:, None])  # (B, C, T', F')
        b, c, t, f = x.shape
        x = self.out(x.transpose(1, 2).reshape(b, t, c * f))
        for _ in range(self.n_convs):
            lengths = (lengths + 2 - 3) // 2 + 1
        return x, lengths


class SinusoidalPositions(nn.Module):
    """Adds the fixed sin/cos position code of "Attention is all you need"."""

    def __init__(self, d_model: int, max_len: int = 10_000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len)[:, None].float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10_000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[: x.size(1)][None]


class FeedForward(nn.Module):
    def __init__(self, d_model: int, mult: int, dropout: float, activation: nn.Module | None = None):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, mult * d_model), activation or nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(mult * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MultiHeadAttention(nn.Module):
    """Standard scaled dot-product attention; `relative=True` adds Transformer-XL style
    relative position bias (Dai et al. 2019), as used by the Conformer."""

    def __init__(self, d_model: int, n_heads: int, dropout: float, relative: bool = False, max_rel: int = 256):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.relative = relative
        self.max_rel = max_rel
        if relative:
            # one learned bias per (head, relative distance), clipped to [-max_rel, max_rel]
            self.rel_bias = nn.Embedding(2 * max_rel + 1, n_heads)

    def forward(
        self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
        key_mask: torch.Tensor | None = None, causal: bool = False,
    ) -> torch.Tensor:
        b, tq, _ = query.shape
        tk = key.size(1)
        q = self.q(query).view(b, tq, self.h, self.dk).transpose(1, 2)  # (B, h, Tq, dk)
        k = self.k(key).view(b, tk, self.h, self.dk).transpose(1, 2)
        v = self.v(value).view(b, tk, self.h, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.dk)  # (B, h, Tq, Tk)
        if self.relative:
            rel = torch.arange(tk, device=query.device)[None, :] - torch.arange(tq, device=query.device)[:, None]
            rel = rel.clamp(-self.max_rel, self.max_rel) + self.max_rel
            scores = scores + self.rel_bias(rel).permute(2, 0, 1)[None]
        if key_mask is not None:
            scores = scores.masked_fill(~key_mask[:, None, None, :], float("-inf"))
        if causal:
            future = torch.triu(torch.ones(tq, tk, dtype=torch.bool, device=query.device), diagonal=1)
            scores = scores.masked_fill(future[None, None], float("-inf"))
        attn = self.drop(torch.softmax(scores, dim=-1))
        out = (attn @ v).transpose(1, 2).reshape(b, tq, self.h * self.dk)
        return self.o(out)


class TransformerBlock(nn.Module):
    """Pre-norm: x + Attn(LN(x)); x + FFN(LN(x))."""

    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)
        self.ffn = FeedForward(cfg.d_model, cfg.ffn_mult, cfg.dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        x = x + self.drop(self.attn(h, h, h, key_mask=mask))
        return x + self.ffn(x)


class ConvModule(nn.Module):
    """Conformer convolution module: pointwise GLU -> depthwise conv -> BN -> SiLU -> pointwise."""

    def __init__(self, d_model: int, kernel: int, dropout: float):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Conv1d(d_model, 2 * d_model, 1)
        self.dw = nn.Conv1d(d_model, d_model, kernel, padding=kernel // 2, groups=d_model)
        self.bn = nn.BatchNorm1d(d_model)
        self.pw2 = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        h = self.norm(x).masked_fill(~mask[..., None], 0.0).transpose(1, 2)  # (B, D, T)
        h = F.glu(self.pw1(h), dim=1).masked_fill(~mask[:, None, :], 0.0)  # the bias would leak into padding
        h = F.silu(self.bn(self.dw(h)))
        h = self.pw2(h).transpose(1, 2)
        return self.drop(h)


class ConformerBlock(nn.Module):
    """x + 1/2 FFN; x + RelMHSA; x + Conv; x + 1/2 FFN; LayerNorm."""

    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ffn1 = FeedForward(cfg.d_model, cfg.ffn_mult, cfg.dropout, nn.SiLU())
        self.norm_attn = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.dropout, relative=True)
        self.drop = nn.Dropout(cfg.dropout)
        self.conv = ConvModule(cfg.d_model, cfg.conv_kernel, cfg.dropout)
        self.ffn2 = FeedForward(cfg.d_model, cfg.ffn_mult, cfg.dropout, nn.SiLU())
        self.norm_out = nn.LayerNorm(cfg.d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = x + 0.5 * self.ffn1(x)
        h = self.norm_attn(x)
        x = x + self.drop(self.attn(h, h, h, key_mask=mask))
        x = x + self.conv(x, mask)
        x = x + 0.5 * self.ffn2(x)
        return self.norm_out(x)


class BiLSTMEncoder(nn.Module):
    def __init__(self, cfg: ModelConfig, n_mels: int):
        super().__init__()
        self.sub = Conv2dSubsampling(n_mels, cfg.d_model, cfg.subsampling_channels, cfg.subsampling_factor)
        self.lstm = nn.LSTM(
            cfg.d_model, cfg.d_model // 2, num_layers=cfg.n_layers, batch_first=True,
            bidirectional=True, dropout=cfg.dropout if cfg.n_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, feats: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x, lengths = self.sub(feats, lengths)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return self.drop(out), lengths


class TransformerEncoder(nn.Module):
    def __init__(self, cfg: ModelConfig, n_mels: int):
        super().__init__()
        self.sub = Conv2dSubsampling(n_mels, cfg.d_model, cfg.subsampling_channels, cfg.subsampling_factor)
        self.pos = SinusoidalPositions(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(TransformerBlock(cfg) for _ in range(cfg.n_layers))
        self.norm = nn.LayerNorm(cfg.d_model)
        self.scale = math.sqrt(cfg.d_model)

    def forward(self, feats: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x, lengths = self.sub(feats, lengths)
        mask = lengths_to_mask(lengths, x.size(1))
        x = self.drop(self.pos(x * self.scale))
        for block in self.blocks:
            x = block(x, mask)
        return self.norm(x), lengths


class ConformerEncoder(nn.Module):
    def __init__(self, cfg: ModelConfig, n_mels: int):
        super().__init__()
        self.sub = Conv2dSubsampling(n_mels, cfg.d_model, cfg.subsampling_channels, cfg.subsampling_factor)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(ConformerBlock(cfg) for _ in range(cfg.n_layers))

    def forward(self, feats: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x, lengths = self.sub(feats, lengths)
        mask = lengths_to_mask(lengths, x.size(1))
        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        return x, lengths


def build_encoder(cfg: ModelConfig, n_mels: int) -> nn.Module:
    return {"bilstm": BiLSTMEncoder, "transformer": TransformerEncoder, "conformer": ConformerEncoder}[cfg.encoder](
        cfg, n_mels
    )

### 4.2 The five recognisers (same as `src/stt/models.py`)

The recognisers. Each one answers the same question, "which text was said?", with a
different way of aligning audio frames (many) to tokens (few):

CTCModel  (Graves et al. 2006)
    The encoder emits one distribution per frame over vocabulary + blank. A frame path such
    as `h h <b> e l l <b> l o` collapses to "hello" (merge repeats, drop blanks). The CTC loss
    sums the probability of EVERY path that collapses to the transcript (dynamic programming,
    `nn.CTCLoss`). Frames are assumed independent given the audio, so the model has no
    language model of its own: cheap, fast, parallel, good with a beam search + LM.

AttentionModel, model="aed"  (Chan et al. 2016 "LAS"; Speech-Transformer)
    An autoregressive decoder emits tokens one at a time, each time looking at the encoder
    output through cross-attention: p(y_i | y_<i, x). Learns an implicit language model, and
    the alignment is soft and free. Teacher forcing at training, beam search at test. Can
    loop or skip words since nothing forces the attention to move monotonically.
AttentionModel, model="las"
    Same idea with an LSTM decoder and location-aware attention (Chorowski et al. 2015):
    the attention also sees where it looked at the previous step (a convolution over the
    previous weights), which pushes it to move forward through the audio.
AttentionModel, model="hybrid"  (Watanabe et al. 2017)
    The CTC head and the attention decoder share the encoder. The CTC loss forces a
    monotonic alignment during training, and at decoding the attention beam is rescored
    by the CTC prefix probability. Best of both; the default in ESPnet.

TransducerModel  (Graves 2012, RNN-T)
    Three parts: the encoder (audio), a prediction network (an LSTM over the previous
    tokens = a small language model) and a joiner that combines encoder frame t with
    prediction state u into a distribution over vocabulary + blank. The transducer loss sums
    over all monotonic paths through the (T x U) lattice. Streaming-friendly (no attention
    over the future), the standard in on-device ASR.

All models take `(feats, feat_lengths, targets, target_lengths)` in `forward` and return
a dict with "loss" (and parts), and expose `encode(feats, lengths)` for decoding.py.

In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn



def add_sos_eos(targets: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Decoder input `<sos> y` and decoder target `y <eos>` (pad = BLANK, ignored in the loss)."""
    b, u = targets.shape
    inp = torch.full((b, u + 1), BLANK, dtype=torch.long, device=targets.device)
    out = torch.full((b, u + 1), BLANK, dtype=torch.long, device=targets.device)
    inp[:, 0] = SOS
    inp[:, 1:] = targets
    out[:, :u] = targets
    out[torch.arange(b), lengths] = EOS
    return inp, out


class CTCModel(nn.Module):
    def __init__(self, cfg: ModelConfig, n_mels: int, vocab_size: int):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = vocab_size
        self.encoder = build_encoder(cfg, n_mels)
        self.ctc_head = nn.Linear(cfg.d_model, vocab_size)

    def encode(self, feats: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encoder(feats, lengths)

    def ctc_log_probs(self, enc: torch.Tensor) -> torch.Tensor:
        return F.log_softmax(self.ctc_head(enc), dim=-1)

    def forward(self, feats, feat_lengths, targets, target_lengths) -> dict[str, torch.Tensor]:
        enc, enc_lengths = self.encode(feats, feat_lengths)
        loss = ctc_loss(self.ctc_log_probs(enc), enc_lengths, targets, target_lengths)
        return {"loss": loss, "ctc": loss.detach()}


def ctc_loss(log_probs: torch.Tensor, enc_lengths, targets, target_lengths) -> torch.Tensor:
    """`log_probs` (B, T, V) -> mean CTC loss per target token (fp32, as the kernel needs)."""
    lp = log_probs.float().transpose(0, 1)  # (T, B, V)
    loss = F.ctc_loss(lp, targets, enc_lengths, target_lengths, blank=BLANK, reduction="sum", zero_infinity=True)
    return loss / target_lengths.sum().clamp(min=1)


class TransformerDecoder(nn.Module):
    """Pre-norm Transformer decoder: causal self-attention over previous tokens, cross-attention
    over the encoder frames, feed-forward; `forward(tokens, enc, enc_mask) -> logits`."""

    def __init__(self, cfg: ModelConfig, vocab_size: int):
        super().__init__()
        d = cfg.dec_dim
        self.embed = nn.Embedding(vocab_size, d)
        self.pos = SinusoidalPositions(d)
        self.drop = nn.Dropout(cfg.dropout)
        self.enc_proj = nn.Linear(cfg.d_model, d) if cfg.d_model != d else nn.Identity()
        self.layers = nn.ModuleList()
        for _ in range(cfg.dec_layers):
            self.layers.append(nn.ModuleDict({
                "n1": nn.LayerNorm(d), "self": MultiHeadAttention(d, cfg.n_heads, cfg.dropout),
                "n2": nn.LayerNorm(d), "cross": MultiHeadAttention(d, cfg.n_heads, cfg.dropout),
                "n3": nn.LayerNorm(d),
                "ffn": nn.Sequential(nn.Linear(d, cfg.ffn_mult * d), nn.ReLU(), nn.Dropout(cfg.dropout),
                                     nn.Linear(cfg.ffn_mult * d, d)),
            }))
        self.norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, vocab_size)
        self.scale = d**0.5

    def forward(self, tokens: torch.Tensor, enc: torch.Tensor, enc_mask: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.pos(self.embed(tokens) * self.scale))
        mem = self.enc_proj(enc)
        for layer in self.layers:
            h = layer["n1"](x)
            x = x + self.drop(layer["self"](h, h, h, causal=True))
            h = layer["n2"](x)
            x = x + self.drop(layer["cross"](h, mem, mem, key_mask=enc_mask))
            x = x + self.drop(layer["ffn"](layer["n3"](x)))
        return self.out(self.norm(x))


class LocationAwareAttention(nn.Module):
    """Additive attention whose score also sees a conv over the previous attention weights
    (Chorowski et al. 2015): e_t = w^T tanh(W s + V h_t + U (f * a_prev)_t + b)."""

    def __init__(self, dec_dim: int, enc_dim: int, attn_dim: int = 128, conv_channels: int = 10, kernel: int = 31):
        super().__init__()
        self.query = nn.Linear(dec_dim, attn_dim, bias=False)
        self.key = nn.Linear(enc_dim, attn_dim)
        self.loc_conv = nn.Conv1d(1, conv_channels, kernel, padding=kernel // 2, bias=False)
        self.loc = nn.Linear(conv_channels, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, state: torch.Tensor, keys: torch.Tensor, enc: torch.Tensor, prev_attn: torch.Tensor,
                mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        loc = self.loc(self.loc_conv(prev_attn[:, None]).transpose(1, 2))  # (B, T, A)
        e = self.v(torch.tanh(self.query(state)[:, None] + keys + loc)).squeeze(-1)  # (B, T)
        e = e.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(e, dim=-1)
        context = torch.bmm(attn[:, None], enc).squeeze(1)  # (B, D)
        return context, attn


class LASDecoder(nn.Module):
    """LSTM decoder with location-aware attention: at each step, input = [embed(y_{i-1}), context],
    then attend, then predict y_i from [state, context]. Stateful: see `step`."""

    def __init__(self, cfg: ModelConfig, vocab_size: int):
        super().__init__()
        d = cfg.dec_dim
        self.embed = nn.Embedding(vocab_size, d)
        self.rnn = nn.LSTMCell(d + cfg.d_model, d)
        self.attn = LocationAwareAttention(d, cfg.d_model)
        self.out = nn.Linear(d + cfg.d_model, vocab_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.enc_dim = cfg.d_model
        self.dec_dim = d

    def init_state(self, enc: torch.Tensor) -> dict:
        b, t, _ = enc.shape
        z = enc.new_zeros(b, self.dec_dim)
        return {"h": z, "c": z.clone(), "context": enc.new_zeros(b, self.enc_dim),
                "attn": enc.new_zeros(b, t), "keys": self.attn.key(enc)}

    def step(self, token: torch.Tensor, state: dict, enc: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, dict]:
        x = torch.cat([self.drop(self.embed(token)), state["context"]], dim=-1)
        h, c = self.rnn(x, (state["h"], state["c"]))
        context, attn = self.attn(h, state["keys"], enc, state["attn"], mask)
        logits = self.out(self.drop(torch.cat([h, context], dim=-1)))
        return logits, {"h": h, "c": c, "context": context, "attn": attn, "keys": state["keys"]}

    def forward(self, tokens: torch.Tensor, enc: torch.Tensor, enc_mask: torch.Tensor) -> torch.Tensor:
        state = self.init_state(enc)
        logits = []
        for i in range(tokens.size(1)):
            step_logits, state = self.step(tokens[:, i], state, enc, enc_mask)
            logits.append(step_logits)
        return torch.stack(logits, dim=1)


class AttentionModel(nn.Module):
    """Encoder + attention decoder (Transformer for aed/hybrid, LSTM for las); hybrid adds a CTC head."""

    def __init__(self, cfg: ModelConfig, n_mels: int, vocab_size: int):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = vocab_size
        self.encoder = build_encoder(cfg, n_mels)
        self.decoder = LASDecoder(cfg, vocab_size) if cfg.model == "las" else TransformerDecoder(cfg, vocab_size)
        self.ctc_head = nn.Linear(cfg.d_model, vocab_size) if cfg.model == "hybrid" else None
        self.ctc_weight = cfg.ctc_weight if cfg.model == "hybrid" else 0.0

    def encode(self, feats: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encoder(feats, lengths)

    def ctc_log_probs(self, enc: torch.Tensor) -> torch.Tensor:
        assert self.ctc_head is not None
        return F.log_softmax(self.ctc_head(enc), dim=-1)

    def forward(self, feats, feat_lengths, targets, target_lengths) -> dict[str, torch.Tensor]:
        enc, enc_lengths = self.encode(feats, feat_lengths)
        mask = lengths_to_mask(enc_lengths, enc.size(1))
        dec_in, dec_out = add_sos_eos(targets, target_lengths)
        logits = self.decoder(dec_in, enc, mask)
        att = F.cross_entropy(
            logits.reshape(-1, self.vocab_size).float(), dec_out.reshape(-1), ignore_index=BLANK,
            label_smoothing=self.cfg.label_smoothing,
        )
        out = {"att": att.detach()}
        with torch.no_grad():
            pred = logits.argmax(-1)
            valid = dec_out != BLANK
            out["token_acc"] = ((pred == dec_out) & valid).sum() / valid.sum().clamp(min=1)
        if self.ctc_head is not None:
            ctc = ctc_loss(self.ctc_log_probs(enc), enc_lengths, targets, target_lengths)
            out["ctc"] = ctc.detach()
            out["loss"] = self.ctc_weight * ctc + (1 - self.ctc_weight) * att
        else:
            out["loss"] = att
        return out


class PredictionNetwork(nn.Module):
    """RNN-T prediction network: an LSTM over the previous non-blank tokens (blank = start)."""

    def __init__(self, cfg: ModelConfig, vocab_size: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, cfg.pred_dim)
        self.lstm = nn.LSTM(cfg.pred_dim, cfg.pred_dim, num_layers=cfg.pred_layers, batch_first=True,
                            dropout=cfg.dropout if cfg.pred_layers > 1 else 0.0)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, tokens: torch.Tensor, state=None) -> tuple[torch.Tensor, tuple]:
        out, state = self.lstm(self.drop(self.embed(tokens)), state)
        return out, state


class Joiner(nn.Module):
    """joint(t, u) = Linear(tanh(W_enc h_t + W_pred g_u)). Broadcasts to (B, T, U, V)."""

    def __init__(self, cfg: ModelConfig, vocab_size: int):
        super().__init__()
        self.enc = nn.Linear(cfg.d_model, cfg.joint_dim)
        self.pred = nn.Linear(cfg.pred_dim, cfg.joint_dim)
        self.out = nn.Linear(cfg.joint_dim, vocab_size)

    def forward(self, enc: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
        return self.out(torch.tanh(self.enc(enc) + self.pred(pred)))


class TransducerModel(nn.Module):
    def __init__(self, cfg: ModelConfig, n_mels: int, vocab_size: int):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = vocab_size
        self.encoder = build_encoder(cfg, n_mels)
        self.predictor = PredictionNetwork(cfg, vocab_size)
        self.joiner = Joiner(cfg, vocab_size)

    def encode(self, feats: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encoder(feats, lengths)

    def forward(self, feats, feat_lengths, targets, target_lengths) -> dict[str, torch.Tensor]:
        try:
            from torchaudio.functional import rnnt_loss
        except ImportError as e:  # pragma: no cover
            raise ImportError("RNN-T training needs torchaudio: pip install 'stt-workbench[rnnt]'") from e
        enc, enc_lengths = self.encode(feats, feat_lengths)
        b, _u = targets.shape
        pred_in = torch.cat([torch.full((b, 1), BLANK, dtype=torch.long, device=targets.device), targets], dim=1)
        pred, _ = self.predictor(pred_in)  # (B, U+1, P)
        logits = self.joiner(enc[:, :, None, :], pred[:, None, :, :])  # (B, T, U+1, V)
        loss = rnnt_loss(
            logits.float(), targets.int(), enc_lengths.int(), target_lengths.int(), blank=BLANK, reduction="sum",
        ) / target_lengths.sum().clamp(min=1)
        return {"loss": loss, "rnnt": loss.detach()}


def build_model(cfg: ModelConfig, n_mels: int, vocab_size: int) -> nn.Module:
    if cfg.model == "ctc":
        return CTCModel(cfg, n_mels, vocab_size)
    if cfg.model in ("aed", "las", "hybrid"):
        return AttentionModel(cfg, n_mels, vocab_size)
    if cfg.model == "rnnt":
        return TransducerModel(cfg, n_mels, vocab_size)
    raise ValueError(cfg.model)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

## 5. Decoding, language model and metrics

### 5.1 n-gram language model (same as `src/stt/lm.py`)

A count-based n-gram language model over token ids, to bias beam search towards likely text.

p(w | h) for history h of the last n-1 tokens, with interpolated absolute discounting
(the simple cousin of Kneser-Ney): mass D is taken from every seen continuation and given
to the lower-order model, recursively down to a uniform distribution.

    p_n(w | h) = max(c(h, w) - D, 0) / c(h)  +  D * N1+(h) / c(h) * p_{n-1}(w | h')

Here N1+(h) is the number of distinct words seen after h. Unseen histories fall back
entirely to the shorter one. The LM scores tokens of whatever tokenizer the model uses
(characters by default), and `<eos>` is a normal token so the LM knows how sentences end.

A character 5-gram trained on the training transcripts is a cheap, honest baseline;
decoding.py adds `lm_weight * log p_lm` to the acoustic score of every extension.

In [ ]:
from __future__ import annotations

import json
import math
from collections import defaultdict
from pathlib import Path



class NGramLM:
    def __init__(self, order: int = 5, vocab_size: int = 32, discount: float = 0.75):
        if order < 1:
            raise ValueError("order must be >= 1")
        self.order = order
        self.vocab_size = vocab_size
        self.discount = discount
        # counts[k][history tuple][word] for history length k
        self.counts: list[dict[tuple, dict[int, int]]] = [defaultdict(lambda: defaultdict(int)) for _ in range(order)]

    def train(self, sequences: list[list[int]]) -> NGramLM:
        for seq in sequences:
            tokens = [SOS] * (self.order - 1) + list(seq) + [EOS]
            for i in range(self.order - 1, len(tokens)):
                w = tokens[i]
                for k in range(self.order):
                    h = tuple(tokens[i - k : i])
                    self.counts[k][h][w] += 1
        return self

    def score(self, history: list[int], token: int) -> float:
        """log p(token | history)."""
        return math.log(max(self.prob(history, token), 1e-12))

    def prob(self, history: list[int], token: int) -> float:
        hist = ([SOS] * (self.order - 1) + list(history))[-(self.order - 1) :] if self.order > 1 else []
        return self._prob(tuple(hist), token, len(hist))

    def _prob(self, h: tuple, w: int, k: int) -> float:
        if k < 0:
            return 1.0 / self.vocab_size
        table = self.counts[k].get(h)
        if not table:
            return self._prob(h[1:] if k > 0 else h, w, k - 1)
        total = sum(table.values())
        n1 = len(table)
        high = max(table.get(w, 0) - self.discount, 0.0) / total
        low = self.discount * n1 / total * self._prob(h[1:] if k > 0 else h, w, k - 1)
        return high + low

    def perplexity(self, sequences: list[list[int]]) -> float:
        logp, n = 0.0, 0
        for seq in sequences:
            hist: list[int] = []
            for w in list(seq) + [EOS]:
                logp += self.score(hist, w)
                hist.append(w)
                n += 1
        return math.exp(-logp / max(n, 1))

    def save(self, path: str | Path) -> None:
        data = {
            "order": self.order, "vocab_size": self.vocab_size, "discount": self.discount,
            "counts": [{" ".join(map(str, h)): dict(t) for h, t in table.items()} for table in self.counts],
        }
        Path(path).write_text(json.dumps(data))

    @classmethod
    def load(cls, path: str | Path) -> NGramLM:
        data = json.loads(Path(path).read_text())
        lm = cls(data["order"], data["vocab_size"], data["discount"])
        for k, table in enumerate(data["counts"]):
            for h, t in table.items():
                key = tuple(int(x) for x in h.split()) if h else ()
                for w, c in t.items():
                    lm.counts[k][key][int(w)] = c
        return lm

### 5.2 Search algorithms (same as `src/stt/decoding.py`)

From model outputs to token sequences: the search algorithms.

CTC
    greedy       argmax per frame, merge repeats, drop blanks. One pass, no search.
    prefix beam  (Hannun et al. 2014) keeps the best `beam` PREFIXES (not frame paths), each
                 with two probabilities: ending in blank (p_b) and ending in a non-blank
                 (p_nb), so that all paths collapsing to the same prefix are merged. An
                 n-gram LM (lm.py) can add `lm_weight * log p_lm(c | prefix)` at each
                 extension, with `word_bonus` per emitted token to counter the LM's bias
                 towards short outputs.

Attention decoders (aed / las / hybrid)
    greedy       feed <sos>, take the argmax token, feed it back, stop at <eos>.
    beam         keep the best `beam` partial sentences by summed log-prob; finished ones
                 (ended with <eos>) are set aside; scores are length-normalised at the end
                 so long sentences are not punished. For hybrid models, the CTC prefix
                 score (Watanabe et al. 2017) of each extension is mixed in:
                 score = (1 - w) * log p_att + w * log p_ctc_prefix, which rejects extensions
                 the CTC alignment finds impossible (skips, loops).

RNN-T
    greedy       for each encoder frame, emit symbols until the joiner says blank
                 (at most `max_symbols` per frame), updating the prediction network.
    beam         "modified" beam search (one symbol per frame at most): at each frame every
                 hypothesis either emits one symbol or blank; hypotheses with the same token
                 sequence are merged by log-sum-exp.

Every function returns token id lists (specials removed), one per utterance; the
tokenizer's `decode` turns them into text.

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field

import torch


NEG_INF = float("-inf")


def logsumexp(*xs: float) -> float:
    m = max(xs)
    if m == NEG_INF:
        return NEG_INF
    return m + math.log(sum(math.exp(x - m) for x in xs))


# ---------------------------------------------------------------------------------------- CTC
def ctc_greedy(log_probs: torch.Tensor, lengths: torch.Tensor) -> list[list[int]]:
    """(B, T, V) log-probs -> token lists. Collapse repeats, remove blanks."""
    best = log_probs.argmax(-1).cpu()
    out = []
    for b in range(best.size(0)):
        seq, prev = [], BLANK
        for t in range(int(lengths[b])):
            tok = int(best[b, t])
            if tok != BLANK and tok != prev:
                seq.append(tok)
            prev = tok
        out.append(seq)
    return out


def ctc_prefix_beam_search(
    log_probs: torch.Tensor, beam: int = 8, lm: NGramLM | None = None, lm_weight: float = 0.5,
    word_bonus: float = 0.0, prune: float = -10.0,
) -> list[int]:
    """One utterance, (T, V) log-probs -> best token list."""
    lp = log_probs.cpu()
    T, V = lp.shape
    # prefix -> (log p_blank, log p_nonblank)
    beams: dict[tuple[int, ...], tuple[float, float]] = {(): (0.0, NEG_INF)}
    lm_cache: dict[tuple[int, ...], float] = {(): 0.0}

    def lm_score(prefix: tuple[int, ...]) -> float:
        if prefix not in lm_cache:
            lm_cache[prefix] = lm_cache[prefix[:-1]] + lm_weight * lm.score(list(prefix[:-1]), prefix[-1]) + word_bonus
        return lm_cache[prefix]

    for t in range(T):
        frame = lp[t]
        candidates = [c for c in range(V) if frame[c] > prune or c == BLANK]
        new: dict[tuple[int, ...], list[float]] = {}

        def add(prefix, pb, pnb, new=new):
            cur = new.setdefault(prefix, [NEG_INF, NEG_INF])
            cur[0] = logsumexp(cur[0], pb)
            cur[1] = logsumexp(cur[1], pnb)

        for prefix, (pb, pnb) in beams.items():
            total = logsumexp(pb, pnb)
            for c in candidates:
                p = float(frame[c])
                if c == BLANK:
                    add(prefix, total + p, NEG_INF)
                elif prefix and c == prefix[-1]:
                    add(prefix, NEG_INF, pnb + p)  # repeat without blank: same prefix
                    add(prefix + (c,), NEG_INF, pb + p)  # repeat after blank: new token
                else:
                    add(prefix + (c,), NEG_INF, total + p)

        def rank(item):
            prefix, (pb, pnb) = item
            s = logsumexp(pb, pnb)
            return s + (lm_score(prefix) if lm is not None else word_bonus * len(prefix))

        beams = dict(sorted(((k, tuple(v)) for k, v in new.items()), key=rank, reverse=True)[:beam])
    best = max(beams.items(), key=lambda kv: logsumexp(*kv[1]) + (lm_score(kv[0]) if lm else 0.0))
    return list(best[0])


class CTCPrefixScorer:
    """CTC prefix probability p(g, ...) for joint CTC/attention decoding (Watanabe et al. 2017).

    For a prefix g, keeps r_n[t] / r_b[t] = log-prob of g at frame t ending in non-blank/blank.
    `extend(state, c)` returns the new state and log p_ctc(g + c), the probability that the
    output STARTS with g + c (any continuation), which attention beam search uses as the
    CTC score of the extension.
    """

    def __init__(self, log_probs: torch.Tensor):
        self.lp = log_probs.cpu()
        self.T = self.lp.size(0)

    def initial(self) -> tuple[torch.Tensor, torch.Tensor, int, float]:
        r_b = torch.cumsum(self.lp[:, BLANK], dim=0)  # all-blank paths
        r_n = torch.full((self.T,), NEG_INF)
        return r_n, r_b, BLANK, 0.0

    def extend(self, state, c: int) -> tuple[tuple, float]:
        r_n, r_b, last, _ = state
        new_n = torch.full((self.T,), NEG_INF)
        new_b = torch.full((self.T,), NEG_INF)
        lp_c = self.lp[:, c]
        lp_b = self.lp[:, BLANK]
        # phi[t-1]: probability of g at t-1 that may be followed by c (no merge with a repeat)
        prev = torch.logaddexp(r_n, r_b) if c != last else r_b
        if last == BLANK and r_n[0] == NEG_INF and r_b[0] != NEG_INF:
            new_n[0] = lp_c[0]  # empty prefix: c may start at frame 0
        for t in range(1, self.T):
            new_n[t] = torch.logaddexp(new_n[t - 1], prev[t - 1]) + lp_c[t]
            new_b[t] = torch.logaddexp(new_n[t - 1], new_b[t - 1]) + lp_b[t]
        # p(g + c, ...) = sum over t of phi[t-1] * p(c | t), plus starting at 0
        psi = torch.logsumexp(torch.cat([prev[:-1] + lp_c[1:], new_n[:1]]), dim=0)
        return (new_n, new_b, c, float(psi)), float(psi)

    def final(self, state) -> float:
        r_n, r_b, _, _ = state
        return float(torch.logaddexp(r_n[-1], r_b[-1]))


# ------------------------------------------------------------------------------ attention
class DecoderInterface:
    """A uniform step API over the two attention decoders (models.py).

    `init(enc, mask)` -> state; `step(state, prefix_tokens)` -> (log_probs (V,), new state).
    The Transformer decoder re-runs on the whole prefix (simple; fine for short outputs);
    the LAS decoder carries its LSTM/attention state along.
    """

    def __init__(self, model, enc: torch.Tensor, mask: torch.Tensor):
        self.model = model
        self.enc = enc  # (1, T, D)
        self.mask = mask
        self.is_las = hasattr(model.decoder, "step")

    def init(self):
        return self.model.decoder.init_state(self.enc) if self.is_las else None

    def step(self, state, prefix: list[int]):
        if self.is_las:
            token = torch.tensor([prefix[-1]], device=self.enc.device)
            logits, state = self.model.decoder.step(token, state, self.enc, self.mask)
            return torch.log_softmax(logits[0].float(), -1), state
        tokens = torch.tensor([prefix], device=self.enc.device)
        logits = self.model.decoder(tokens, self.enc, self.mask)
        return torch.log_softmax(logits[0, -1].float(), -1), None


@dataclass(order=True)
class Hypothesis:
    score: float
    tokens: list[int] = field(compare=False)
    state: object = field(compare=False, default=None)
    ctc_state: object = field(compare=False, default=None)
    att_score: float = field(compare=False, default=0.0)


@torch.no_grad()
def attention_greedy(model, enc: torch.Tensor, mask: torch.Tensor, max_len: int) -> list[int]:
    dec = DecoderInterface(model, enc, mask)
    state, prefix = dec.init(), [SOS]
    for _ in range(max_len):
        lp, state = dec.step(state, prefix)
        tok = int(lp.argmax())
        if tok == EOS:
            break
        prefix.append(tok)
    return [t for t in prefix[1:] if t >= len(SPECIALS)]


@torch.no_grad()
def attention_beam_search(
    model, enc: torch.Tensor, mask: torch.Tensor, max_len: int, beam: int = 4, ctc_weight: float = 0.0,
    ctc_log_probs: torch.Tensor | None = None, lm: NGramLM | None = None, lm_weight: float = 0.3,
    length_penalty: float = 1.0,
) -> list[int]:
    """One utterance (enc (1, T, D)). Joint CTC/attention when ctc_weight > 0 and ctc_log_probs given."""
    dec = DecoderInterface(model, enc, mask)
    scorer = CTCPrefixScorer(ctc_log_probs) if (ctc_weight > 0 and ctc_log_probs is not None) else None
    start = Hypothesis(0.0, [SOS], dec.init(), scorer.initial() if scorer else None)
    live, done = [start], []
    for _ in range(max_len):
        candidates = []
        for hyp in live:
            lp, state = dec.step(hyp.state, hyp.tokens)
            topk = torch.topk(lp, min(beam * 2, lp.numel()))
            for score, tok in zip(topk.values.tolist(), topk.indices.tolist()):
                if tok in (SOS, BLANK):
                    continue
                if tok == EOS:
                    total = hyp.score + (1 - ctc_weight) * score
                    if scorer is not None:
                        total += ctc_weight * (scorer.final(hyp.ctc_state) - hyp.ctc_state[3])
                    if lm is not None:
                        total += lm_weight * lm.score(hyp.tokens[1:], EOS)
                    done.append(Hypothesis(total, hyp.tokens[1:], None))
                    continue
                total = hyp.score + (1 - ctc_weight) * score
                ctc_state = None
                if scorer is not None:
                    ctc_state, psi = scorer.extend(hyp.ctc_state, tok)
                    total += ctc_weight * (psi - hyp.ctc_state[3])  # prefix score increment
                if lm is not None:
                    total += lm_weight * lm.score(hyp.tokens[1:], tok)
                candidates.append(Hypothesis(total, hyp.tokens + [tok], state, ctc_state))
        if not candidates:
            break
        live = sorted(candidates, reverse=True)[:beam]
        if done and len(done) >= beam:
            best_done = max(h.score / (len(h.tokens) + 1) ** length_penalty for h in done)
            if all(h.score / (len(h.tokens)) ** length_penalty < best_done for h in live):
                break
    if not done:
        done = [Hypothesis(h.score, h.tokens[1:]) for h in live]
    best = max(done, key=lambda h: h.score / (len(h.tokens) + 1) ** length_penalty)
    return [t for t in best.tokens if t >= len(SPECIALS)]


# ------------------------------------------------------------------------------------ RNN-T
@torch.no_grad()
def rnnt_greedy(model, enc: torch.Tensor, max_symbols: int = 4) -> list[int]:
    """One utterance, enc (1, T, D)."""
    tokens: list[int] = []
    pred_out, state = model.predictor(torch.tensor([[BLANK]], device=enc.device))
    for t in range(enc.size(1)):
        for _ in range(max_symbols):
            logits = model.joiner(enc[:, t], pred_out[:, -1])
            tok = int(logits.argmax(-1))
            if tok == BLANK:
                break
            tokens.append(tok)
            pred_out, state = model.predictor(torch.tensor([[tok]], device=enc.device), state)
    return [t for t in tokens if t >= len(SPECIALS)]


@torch.no_grad()
def rnnt_beam_search(model, enc: torch.Tensor, beam: int = 4, lm: NGramLM | None = None, lm_weight: float = 0.3) -> list[int]:
    """Modified beam search (at most one symbol per frame), merging identical token sequences."""
    pred_out, state = model.predictor(torch.tensor([[BLANK]], device=enc.device))
    hyps: dict[tuple[int, ...], Hypothesis] = {(): Hypothesis(0.0, [], (pred_out, state))}
    for t in range(enc.size(1)):
        new: dict[tuple[int, ...], Hypothesis] = {}
        for key, hyp in hyps.items():
            pred_out, state = hyp.state
            lp = torch.log_softmax(model.joiner(enc[:, t], pred_out[:, -1])[0].float(), -1)
            # blank: same tokens, same predictor state
            blank_score = hyp.score + float(lp[BLANK])
            if key in new:
                new[key].score = logsumexp(new[key].score, blank_score)
            else:
                new[key] = Hypothesis(blank_score, hyp.tokens, hyp.state)
            topk = torch.topk(lp, min(beam, lp.numel()))
            for score, tok in zip(topk.values.tolist(), topk.indices.tolist()):
                if tok == BLANK or tok < len(SPECIALS):
                    continue
                total = hyp.score + score + (lm_weight * lm.score(hyp.tokens, tok) if lm else 0.0)
                new_key = key + (tok,)
                if new_key in new:
                    new[new_key].score = logsumexp(new[new_key].score, total)
                else:
                    new[new_key] = Hypothesis(total, hyp.tokens + [tok], None)
        # run the predictor once per new token sequence (lazy state)
        kept = sorted(new.values(), reverse=True)[:beam]
        for h in kept:
            if h.state is None:
                parent = hyps.get(tuple(h.tokens[:-1]))
                p_out, p_state = model.predictor(torch.tensor([[h.tokens[-1]]], device=enc.device),
                                                 parent.state[1] if parent else None)
                h.state = (p_out, p_state)
        hyps = {tuple(h.tokens): h for h in kept}
    best = max(hyps.values())
    return best.tokens


# ---------------------------------------------------------------------------------- entry
@torch.no_grad()
def decode_batch(
    model, feats: torch.Tensor, lengths: torch.Tensor, method: str = "greedy", beam: int = 4,
    lm: NGramLM | None = None, lm_weight: float = 0.3, max_len_ratio: float = 1.0,
) -> list[list[int]]:
    """Decode a padded batch with whatever the model is. Returns token id lists."""
    cfg = model.cfg
    enc, enc_lengths = model.encode(feats, lengths)
    if cfg.model == "ctc":
        lp = model.ctc_log_probs(enc)
        if method == "greedy":
            return ctc_greedy(lp, enc_lengths)
        return [ctc_prefix_beam_search(lp[b, : int(enc_lengths[b])], beam, lm, lm_weight) for b in range(lp.size(0))]
    out = []
    for b in range(enc.size(0)):
        n = int(enc_lengths[b])
        e = enc[b : b + 1, :n]
        if cfg.model == "rnnt":
            out.append(rnnt_greedy(model, e) if method == "greedy" else rnnt_beam_search(model, e, beam, lm, lm_weight))
            continue
        mask = torch.ones(1, n, dtype=torch.bool, device=enc.device)
        max_len = max(4, int(n * max_len_ratio))
        if method == "greedy":
            out.append(attention_greedy(model, e, mask, max_len))
        else:
            ctc_lp = model.ctc_log_probs(e)[0] if cfg.model == "hybrid" else None
            out.append(attention_beam_search(
                model, e, mask, max_len, beam, ctc_weight=cfg.ctc_weight if cfg.model == "hybrid" else 0.0,
                ctc_log_probs=ctc_lp, lm=lm, lm_weight=lm_weight,
            ))
    return out

### 5.3 WER / CER (same as `src/stt/metrics.py`)

Word error rate (WER) and character error rate (CER), with the alignment behind them.

WER = (S + D + I) / N over the reference words: substitutions, deletions and insertions
found by the Levenshtein alignment that needs the fewest edits, divided by the number of
reference words. CER is the same thing over characters (spaces included). Both can exceed
100% (many insertions). Error rates are computed over the whole test set (sum of edits
over sum of reference lengths), not averaged per utterance.

`stt-eval` compares two files (reference and hypothesis manifests / text files) and prints
WER, CER and the worst utterances with their alignments, e.g.

    REF: the cat  sat on the mat
    HYP: the cat  sat in *** mat
                      S   D

In [ ]:
from __future__ import annotations

import argparse
import json
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

from typing_extensions import Self


@dataclass
class EditStats:
    substitutions: int = 0
    deletions: int = 0
    insertions: int = 0
    ref_length: int = 0

    @property
    def errors(self) -> int:
        return self.substitutions + self.deletions + self.insertions

    @property
    def rate(self) -> float:
        return self.errors / self.ref_length if self.ref_length else (1.0 if self.errors else 0.0)

    def __iadd__(self, other: EditStats) -> Self:
        self.substitutions += other.substitutions
        self.deletions += other.deletions
        self.insertions += other.insertions
        self.ref_length += other.ref_length
        return self


def edit_alignment(ref: Sequence, hyp: Sequence) -> list[tuple[str, object, object]]:
    """Levenshtein alignment: list of (op, ref_item, hyp_item) with op in {"=", "S", "D", "I"}."""
    n, m = len(ref), len(hyp)
    cost = [[0] * (m + 1) for _ in range(n + 1)]
    back = [[""] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        cost[i][0], back[i][0] = i, "D"
    for j in range(1, m + 1):
        cost[0][j], back[0][j] = j, "I"
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            same = ref[i - 1] == hyp[j - 1]
            sub = cost[i - 1][j - 1] + (0 if same else 1)
            dele = cost[i - 1][j] + 1
            ins = cost[i][j - 1] + 1
            best = min(sub, dele, ins)
            cost[i][j] = best
            back[i][j] = ("=" if same else "S") if best == sub else ("D" if best == dele else "I")
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        op = back[i][j]
        if op in ("=", "S"):
            ops.append((op, ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif op == "D":
            ops.append(("D", ref[i - 1], None))
            i -= 1
        else:
            ops.append(("I", None, hyp[j - 1]))
            j -= 1
    return ops[::-1]


def edit_stats(ref: Sequence, hyp: Sequence) -> EditStats:
    ops = edit_alignment(ref, hyp)
    return EditStats(
        substitutions=sum(op == "S" for op, _, _ in ops),
        deletions=sum(op == "D" for op, _, _ in ops),
        insertions=sum(op == "I" for op, _, _ in ops),
        ref_length=len(ref),
    )


def wer(refs: list[str], hyps: list[str]) -> float:
    total = EditStats()
    for r, h in zip(refs, hyps, strict=True):
        total += edit_stats(r.split(), h.split())
    return total.rate


def cer(refs: list[str], hyps: list[str]) -> float:
    total = EditStats()
    for r, h in zip(refs, hyps, strict=True):
        total += edit_stats(list(r), list(h))
    return total.rate


def error_report(refs: list[str], hyps: list[str]) -> dict:
    """WER/CER plus the counts behind them and per-utterance WER."""
    words, chars, per_utt = EditStats(), EditStats(), []
    for r, h in zip(refs, hyps, strict=True):
        w = edit_stats(r.split(), h.split())
        words += w
        chars += edit_stats(list(r), list(h))
        per_utt.append(w.rate)
    return {
        "wer": words.rate, "cer": chars.rate, "n_utterances": len(refs),
        "word_substitutions": words.substitutions, "word_deletions": words.deletions,
        "word_insertions": words.insertions, "ref_words": words.ref_length, "per_utterance_wer": per_utt,
    }


def format_alignment(ref: str, hyp: str) -> str:
    """Three aligned lines (REF / HYP / ops) for one utterance, word level."""
    ops = edit_alignment(ref.split(), hyp.split())
    cells = []
    for op, r, h in ops:
        r = r or "***"
        h = h or "***"
        width = max(len(r), len(h))
        cells.append((r.ljust(width), h.ljust(width), (op if op != "=" else " ").ljust(width)))
    return "\n".join(f"{label}: " + " ".join(c[i] for c in cells) for i, label in enumerate(("REF", "HYP", "   ")))


def read_texts(path: Path) -> list[str]:
    """A .jsonl manifest (field "text") or a plain text file (one utterance per line)."""
    lines = path.read_text().splitlines()
    if path.suffix == ".jsonl":
        return [json.loads(line)["text"] for line in lines if line.strip()]
    return lines


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="WER / CER between a reference and a hypothesis file")
    p.add_argument("reference", type=Path, help=".jsonl manifest or text file")
    p.add_argument("hypothesis", type=Path, help=".jsonl (field 'text') or text file, same order")
    p.add_argument("--worst", type=int, default=5, help="print the N worst utterances")
    p.add_argument("--json", type=Path, default=None, help="write the report here")
    args = p.parse_args(argv)
    refs, hyps = read_texts(args.reference), read_texts(args.hypothesis)
    report = error_report(refs, hyps)
    print(f"WER {100 * report['wer']:.2f}%  CER {100 * report['cer']:.2f}%  ({report['n_utterances']} utterances, "
          f"S={report['word_substitutions']} D={report['word_deletions']} I={report['word_insertions']} "
          f"N={report['ref_words']})")
    worst = sorted(range(len(refs)), key=lambda i: -report["per_utterance_wer"][i])[: args.worst]
    for i in worst:
        print(f"\n[{i}] WER {100 * report['per_utterance_wer'][i]:.0f}%\n{format_alignment(refs[i], hyps[i])}")
    if args.json:
        args.json.write_text(json.dumps(report, indent=1))

## 6. Training code

### 6.1 Trainer (same as `src/stt/train.py`) (same as `src/stt/train.py`)

Training loop shared by every approach: `stt-train --model ctc --encoder conformer ...`.

What happens each step:
1. a bucketed batch of log-mel features and token targets (data.py);
2. the model computes its own loss (CTC, cross-entropy, joint, or transducer; models.py)
   under autocast if `--precision bf16|fp16` (the sequence losses themselves run in fp32);
3. gradient clipping, AdamW, linear warm-up then cosine decay of the learning rate.

Every `eval_every` steps the validation manifest is decoded with the configured decoder
(greedy by default; `--decode beam`) and WER / CER are computed (metrics.py). `last.pt` is
written every evaluation, `best.pt` when the selected metric improves. Checkpoints hold
the three configs and the tokenizer, so `load_checkpoint(path)` rebuilds everything.
`--time-budget-hours` stops cleanly before a Kaggle/Colab session ends; `--resume` (default)
continues from `last.pt`.

Curves go to `<output_dir>/log.csv` (step, loss, lr, wer, cer, ...) and the final numbers
to `<output_dir>/results.json`.

In [ ]:
from __future__ import annotations

import argparse
import csv
import dataclasses
import json
import math
import random
import time
from contextlib import nullcontext
from pathlib import Path
from typing import Any

import numpy as np
import torch



def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def pick_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def lr_at(step: int, cfg: TrainConfig) -> float:
    """Linear warm-up to `lr`, then cosine decay to `min_lr` at max_steps."""
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / cfg.warmup_steps
    progress = min(1.0, (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps))
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * progress))


def save_checkpoint(path: Path, model, optimizer, step: int, best: float, data_cfg, model_cfg, train_cfg, tokenizer,
                    history: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model": model.state_dict(), "optimizer": optimizer.state_dict() if optimizer else None, "step": step,
        "best": best, "data_config": config_to_dict(data_cfg), "model_config": config_to_dict(model_cfg),
        "train_config": config_to_dict(train_cfg), "tokenizer": tokenizer.to_dict(), "history": history,
    }, path)


def load_checkpoint(path: str | Path, device: str | torch.device = "cpu") -> dict[str, Any]:
    """Rebuild model + tokenizer + configs from a checkpoint. Returns a dict with those keys."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    data_cfg = config_from_dict(DataConfig, ckpt["data_config"])
    model_cfg = config_from_dict(ModelConfig, ckpt["model_config"])
    train_cfg = config_from_dict(TrainConfig, ckpt["train_config"])
    tokenizer = tokenizer_from_dict(ckpt["tokenizer"])
    model = build_model(model_cfg, data_cfg.n_mels, tokenizer.vocab_size)
    model.load_state_dict(ckpt["model"])
    model.to(device).eval()
    return {"model": model, "tokenizer": tokenizer, "data_config": data_cfg, "model_config": model_cfg,
            "train_config": train_cfg, "step": ckpt["step"], "best": ckpt["best"], "history": ckpt.get("history", []),
            "optimizer": ckpt.get("optimizer")}


@torch.no_grad()
def evaluate(model, loader, tokenizer, device, method: str = "greedy", beam: int = 4, max_batches: int | None = None,
             lm=None, lm_weight: float = 0.3) -> dict:
    """Loss + WER/CER over the loader; also returns a few example (ref, hyp) pairs."""
    model.eval()
    refs, hyps, losses = [], [], []
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        feats = batch["feats"].to(device)
        out = model(feats, batch["feat_lengths"].to(device), batch["targets"].to(device), batch["target_lengths"].to(device))
        losses.append(float(out["loss"]))
        ids = decode_batch(model, feats, batch["feat_lengths"].to(device), method, beam, lm, lm_weight)
        hyps += [tokenizer.decode(seq) for seq in ids]
        refs += batch["texts"]
    model.train()
    report = error_report(refs, hyps) if refs else {"wer": float("nan"), "cer": float("nan")}
    examples = list(zip(refs[:5], hyps[:5]))
    return {"loss": float(np.mean(losses)) if losses else float("nan"), "wer": report["wer"], "cer": report["cer"],
            "examples": examples, "n": len(refs)}


def print_flush(*args) -> None:
    print(*args, flush=True)


def train(data_cfg: DataConfig, model_cfg: ModelConfig, train_cfg: TrainConfig, log=print_flush) -> dict:
    set_seed(train_cfg.seed)
    device = pick_device(train_cfg.device)
    out_dir = Path(train_cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_utts = read_manifest(Path(data_cfg.data_dir) / data_cfg.train_manifest, data_cfg.lowercase)
    val_utts = read_manifest(Path(data_cfg.data_dir) / data_cfg.val_manifest, data_cfg.lowercase)
    last = out_dir / "last.pt"
    resume = train_cfg.resume and last.exists()
    if resume:
        state = load_checkpoint(last, device)
        tokenizer, model = state["tokenizer"], state["model"]
        step, best, history = state["step"], state["best"], state["history"]
        log(f"resumed from {last} at step {step}")
    else:
        tokenizer = build_tokenizer(data_cfg, train_utts)
        model = build_model(model_cfg, data_cfg.n_mels, tokenizer.vocab_size).to(device)
        step, best, history = 0, float("inf"), []
    model.train()
    log(f"model {model_cfg.model}/{model_cfg.encoder}: {count_parameters(model) / 1e6:.2f} M parameters, "
        f"vocab {tokenizer.vocab_size}, {len(train_utts)} train / {len(val_utts)} val utterances, device {device}")

    train_ds = SpeechDataset(data_cfg, train_utts, tokenizer, train=True, seed=train_cfg.seed)
    val_ds = SpeechDataset(data_cfg, val_utts, tokenizer, train=False)
    train_loader = make_loader(train_ds, train_cfg.batch_size, True, train_cfg.num_workers, train_cfg.seed)
    val_loader = make_loader(val_ds, train_cfg.batch_size, False, train_cfg.num_workers)

    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (no_decay if p.ndim < 2 or "norm" in name or "bias" in name else decay).append(p)
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": train_cfg.weight_decay},
                                   {"params": no_decay, "weight_decay": 0.0}], lr=train_cfg.lr, betas=(0.9, 0.98))
    if resume and state["optimizer"]:
        optimizer.load_state_dict(state["optimizer"])

    use_amp = device.type == "cuda" and train_cfg.precision != "fp32"
    amp_dtype = torch.bfloat16 if train_cfg.precision == "bf16" else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and train_cfg.precision == "fp16")
    autocast = torch.autocast("cuda", dtype=amp_dtype) if use_amp else nullcontext()

    log_path = out_dir / "log.csv"
    fieldnames = ["step", "epoch", "loss", "lr", "grad_norm", "val_loss", "wer", "cer", "elapsed_s"]
    if not resume or not log_path.exists():
        with log_path.open("w", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writeheader()

    def write_log(row: dict) -> None:
        with log_path.open("a", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow({k: row.get(k, "") for k in fieldnames})

    def run_eval() -> dict:
        res = evaluate(model, val_loader, tokenizer, device, train_cfg.decode, train_cfg.beam_size, train_cfg.eval_batches)
        log(f"  val loss {res['loss']:.3f}  WER {100 * res['wer']:.1f}%  CER {100 * res['cer']:.1f}%  ({res['n']} utts)")
        for ref, hyp in res["examples"][:2]:
            log(f"    ref: {ref!r}\n    hyp: {hyp!r}")
        return res

    t0 = time.time()
    epoch = step // max(1, len(train_loader))
    running: dict[str, float] = {}
    n_running = 0
    stop = False
    budget_s = train_cfg.time_budget_hours * 3600 if train_cfg.time_budget_hours else None
    final: dict = {}
    while not stop:
        train_loader.batch_sampler.set_epoch(epoch)
        for batch in train_loader:
            lr = lr_at(step, train_cfg)
            for g in optimizer.param_groups:
                g["lr"] = lr
            with autocast:
                out = model(batch["feats"].to(device), batch["feat_lengths"].to(device),
                            batch["targets"].to(device), batch["target_lengths"].to(device))
            loss = out["loss"]
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip))
            if math.isfinite(grad_norm):
                scaler.step(optimizer)
            scaler.update()
            step += 1
            for k, v in out.items():
                running[k] = running.get(k, 0.0) + float(v.detach())
            n_running += 1

            if step % train_cfg.log_every == 0:
                parts = "  ".join(f"{k} {v / n_running:.3f}" for k, v in running.items())
                log(f"step {step}  {parts}  lr {lr:.2e}  gn {grad_norm:.1f}  {time.time() - t0:.0f}s")
                write_log({"step": step, "epoch": epoch, "loss": running["loss"] / n_running, "lr": lr,
                           "grad_norm": grad_norm, "elapsed_s": time.time() - t0})
                running, n_running = {}, 0

            out_of_time = budget_s is not None and time.time() - t0 > budget_s
            done = step >= train_cfg.max_steps or out_of_time
            if step % train_cfg.eval_every == 0 or done:
                res = run_eval()
                history.append({"step": step, **{k: res[k] for k in ("loss", "wer", "cer")}})
                write_log({"step": step, "epoch": epoch, "val_loss": res["loss"], "wer": res["wer"], "cer": res["cer"],
                           "elapsed_s": time.time() - t0})
                metric = res[train_cfg.select_by]
                if metric < best:
                    best = metric
                    save_checkpoint(out_dir / "best.pt", model, None, step, best, data_cfg, model_cfg, train_cfg,
                                    tokenizer, history)
                    log(f"  new best {train_cfg.select_by} {best:.4f} -> best.pt")
                save_checkpoint(last, model, optimizer, step, best, data_cfg, model_cfg, train_cfg, tokenizer, history)
                final = {"step": step, "val_loss": res["loss"], "wer": res["wer"], "cer": res["cer"],
                         f"best_{train_cfg.select_by}": best, "train_seconds": time.time() - t0,
                         "parameters": count_parameters(model), "model": model_cfg.model, "encoder": model_cfg.encoder}
                (out_dir / "results.json").write_text(json.dumps(final, indent=1))
            if done:
                stop = True
                break
        epoch += 1
        if train_cfg.max_epochs is not None and epoch >= train_cfg.max_epochs:
            stop = True
    if not final:  # max_steps reached before the first eval (should not happen) -> evaluate anyway
        res = run_eval()
        final = {"step": step, "wer": res["wer"], "cer": res["cer"], "val_loss": res["loss"]}
    log(f"done: {json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in final.items()})}")
    return final


def build_parser() -> argparse.ArgumentParser:
    d, m, t = DataConfig(), ModelConfig(), TrainConfig()
    p = argparse.ArgumentParser(description="Train a speech-to-text model",
                                formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    data = p.add_argument_group("data")
    data.add_argument("--data-dir", type=Path, default=d.data_dir, help="folder with train.jsonl and val.jsonl")
    data.add_argument("--train-manifest", default=d.train_manifest)
    data.add_argument("--val-manifest", default=d.val_manifest)
    data.add_argument("--sample-rate", type=int, default=d.sample_rate)
    data.add_argument("--n-mels", type=int, default=d.n_mels)
    data.add_argument("--tokenizer", choices=["char", "bpe"], default=d.tokenizer)
    data.add_argument("--bpe-model", type=Path, default=None)
    data.add_argument("--no-specaugment", action="store_true")
    data.add_argument("--max-duration-s", type=float, default=d.max_duration_s)
    model = p.add_argument_group("model")
    model.add_argument("--model", choices=MODEL_TYPES, default=m.model)
    model.add_argument("--encoder", choices=ENCODER_TYPES, default=m.encoder)
    model.add_argument("--d-model", type=int, default=m.d_model)
    model.add_argument("--n-heads", type=int, default=m.n_heads)
    model.add_argument("--n-layers", type=int, default=m.n_layers)
    model.add_argument("--ffn-mult", type=int, default=m.ffn_mult)
    model.add_argument("--conv-kernel", type=int, default=m.conv_kernel)
    model.add_argument("--subsampling-factor", type=int, choices=[2, 4], default=m.subsampling_factor)
    model.add_argument("--dropout", type=float, default=m.dropout)
    model.add_argument("--dec-layers", type=int, default=m.dec_layers)
    model.add_argument("--dec-dim", type=int, default=m.dec_dim)
    model.add_argument("--label-smoothing", type=float, default=m.label_smoothing)
    model.add_argument("--ctc-weight", type=float, default=m.ctc_weight, help="hybrid: weight of the CTC loss")
    model.add_argument("--pred-dim", type=int, default=m.pred_dim)
    model.add_argument("--joint-dim", type=int, default=m.joint_dim)
    tr = p.add_argument_group("training")
    tr.add_argument("--output-dir", type=Path, default=t.output_dir)
    tr.add_argument("--batch-size", type=int, default=t.batch_size)
    tr.add_argument("--max-steps", type=int, default=t.max_steps)
    tr.add_argument("--max-epochs", type=int, default=None)
    tr.add_argument("--lr", type=float, default=t.lr)
    tr.add_argument("--warmup-steps", type=int, default=t.warmup_steps)
    tr.add_argument("--weight-decay", type=float, default=t.weight_decay)
    tr.add_argument("--grad-clip", type=float, default=t.grad_clip)
    tr.add_argument("--precision", choices=["fp32", "bf16", "fp16"], default=t.precision)
    tr.add_argument("--num-workers", type=int, default=t.num_workers)
    tr.add_argument("--device", default=t.device)
    tr.add_argument("--seed", type=int, default=t.seed)
    tr.add_argument("--log-every", type=int, default=t.log_every)
    tr.add_argument("--eval-every", type=int, default=t.eval_every)
    tr.add_argument("--eval-batches", type=int, default=None)
    tr.add_argument("--decode", choices=DECODER_METHODS, default=t.decode, help="validation decoder")
    tr.add_argument("--beam-size", type=int, default=t.beam_size)
    tr.add_argument("--select-by", choices=["wer", "cer", "loss"], default=t.select_by)
    tr.add_argument("--time-budget-hours", type=float, default=None)
    tr.add_argument("--no-resume", action="store_true")
    return p


def configs_from_args(args: argparse.Namespace) -> tuple[DataConfig, ModelConfig, TrainConfig]:
    data_cfg = DataConfig(
        data_dir=args.data_dir, train_manifest=args.train_manifest, val_manifest=args.val_manifest,
        sample_rate=args.sample_rate, n_mels=args.n_mels, tokenizer=args.tokenizer, bpe_model=args.bpe_model,
        specaugment=not args.no_specaugment, max_duration_s=args.max_duration_s,
    )
    model_cfg = ModelConfig(
        model=args.model, encoder=args.encoder, d_model=args.d_model, n_heads=args.n_heads, n_layers=args.n_layers,
        ffn_mult=args.ffn_mult, conv_kernel=args.conv_kernel, subsampling_factor=args.subsampling_factor,
        dropout=args.dropout, dec_layers=args.dec_layers, dec_dim=args.dec_dim, label_smoothing=args.label_smoothing,
        ctc_weight=args.ctc_weight, pred_dim=args.pred_dim, joint_dim=args.joint_dim,
    )
    train_cfg = TrainConfig(
        output_dir=args.output_dir, batch_size=args.batch_size, max_steps=args.max_steps, max_epochs=args.max_epochs,
        lr=args.lr, warmup_steps=args.warmup_steps, weight_decay=args.weight_decay, grad_clip=args.grad_clip,
        precision=args.precision, num_workers=args.num_workers, device=args.device, seed=args.seed,
        log_every=args.log_every, eval_every=args.eval_every, eval_batches=args.eval_batches, decode=args.decode,
        beam_size=args.beam_size, select_by=args.select_by, time_budget_hours=args.time_budget_hours,
        resume=not args.no_resume,
    )
    return data_cfg, model_cfg, train_cfg


def main(argv: list[str] | None = None) -> None:
    args = build_parser().parse_args(argv)
    data_cfg, model_cfg, train_cfg = configs_from_args(args)
    for cfg in (data_cfg, model_cfg, train_cfg):
        print(f"{type(cfg).__name__}: {dataclasses.asdict(cfg)}")
    train(data_cfg, model_cfg, train_cfg)

## 7. Train

Runs the trainer with the settings of section 2 and draws the curves (`log.csv`) after
every evaluation. Re-running this cell after a disconnect resumes from `last.pt`.
`MODEL = "wav2vec2"` / `"whisper"` skip this section: see section 8.

In [ ]:
import csv
import time

from IPython.display import clear_output


def read_log(path):
    rows = list(csv.DictReader(open(path))) if Path(path).exists() else []
    train = [(int(r["step"]), float(r["loss"])) for r in rows if r["loss"]]
    val = [(int(r["step"]), float(r["wer"]), float(r["cer"]), float(r["val_loss"])) for r in rows if r["wer"]]
    return train, val


def plot_curves(path):
    train, val = read_log(path)
    if not train:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(*zip(*train), label="train loss")
    if val:
        axes[0].plot([v[0] for v in val], [v[3] for v in val], "o-", label="val loss")
        axes[1].plot([v[0] for v in val], [100 * v[1] for v in val], "o-", label="WER %")
        axes[1].plot([v[0] for v in val], [100 * v[2] for v in val], "s-", label="CER %")
        axes[1].set_ylim(0, min(110, max(5, 100 * max(max(v[1], v[2]) for v in val) * 1.1)))
    for ax in axes:
        ax.legend()
        ax.set_xlabel("step")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


lines = []


def notebook_log(*args):
    text = " ".join(str(a) for a in args)
    lines.append(text)
    if text.startswith("  val loss"):
        clear_output(wait=True)
        plot_curves(OUTPUT_DIR / "log.csv")
        print("\n".join(lines[-12:]))
    else:
        print(text)


if MODEL in MODEL_TYPES:
    t0 = time.time()
    results = train(data_config, model_config, train_config, log=notebook_log)
    print(f"finished in {(time.time() - t0) / 60:.1f} min:", results)
else:
    print("pretrained model selected: go to section 8")

## 8. Pretrained fine-tuning (wav2vec2 / Whisper)

### 8.1 Wrappers (same as `src/stt/pretrained.py`)

Fine-tuning pretrained speech models from Hugging Face (`pip install 'stt-workbench[pretrained]'`).

The models in models.py learn everything from the training set. With little data, a model
that already knows what speech sounds like does far better:

wav2vec2 + CTC  (Baevski et al. 2020)
    A convolutional waveform encoder + Transformer, pretrained by contrastive prediction of
    masked latent frames on 1000s of hours of unlabelled audio. Fine-tuning adds a linear
    CTC head over characters on top and trains with the CTC loss: the same objective as
    `CTCModel`, with a much better encoder. `Wav2Vec2Fine` wraps `Wav2Vec2ForCTC` to the
    repo's (feats, lengths, targets, target_lengths) interface, except that it eats raw
    16 kHz waveforms, not log-mels.

Whisper  (Radford et al. 2022)
    An attention encoder-decoder (like `aed`) trained on 680 000 h of weakly labelled audio,
    with its own tokenizer and log-mel front end. Fine-tuning = continuing its cross-entropy
    training on your transcripts. `WhisperFine` wraps `WhisperForConditionalGeneration`.

Both wrappers are thin: `forward(...)` -> {"loss": ...}, `transcribe(waveforms)` -> texts.
Training them is a notebook job (GPU, model downloads); this module only holds the code
so that the notebook and the package share it. `freeze_feature_encoder` keeps the
convolutional front end fixed, which is standard for small datasets.

In [ ]:
from __future__ import annotations

import torch
from torch import nn



class Wav2Vec2Fine(nn.Module):
    """wav2vec2 with a fresh CTC head over the repo's character vocabulary (blank = 0)."""

    def __init__(self, tokenizer, name: str = "facebook/wav2vec2-base", freeze_feature_encoder: bool = True,
                 config_overrides: dict | None = None, model=None):
        super().__init__()
        self.tokenizer = tokenizer
        if model is None:
            from transformers import Wav2Vec2ForCTC

            model = Wav2Vec2ForCTC.from_pretrained(
                name, vocab_size=tokenizer.vocab_size, pad_token_id=0, ctc_loss_reduction="sum",
                ignore_mismatched_sizes=True, **(config_overrides or {}),
            )
        self.model = model
        if freeze_feature_encoder and hasattr(self.model, "freeze_feature_encoder"):
            self.model.freeze_feature_encoder()

    def forward(self, waves: torch.Tensor, wave_lengths: torch.Tensor, targets: torch.Tensor, target_lengths: torch.Tensor):
        mask = torch.arange(waves.size(1), device=waves.device)[None] < wave_lengths[:, None]
        labels = targets.clone()
        labels[torch.arange(labels.size(1), device=labels.device)[None] >= target_lengths[:, None]] = -100
        out = self.model(input_values=waves, attention_mask=mask.long(), labels=labels)
        return {"loss": out.loss / target_lengths.sum().clamp(min=1)}

    @torch.no_grad()
    def transcribe(self, waves: torch.Tensor, wave_lengths: torch.Tensor) -> list[str]:
        from stt.decoding import ctc_greedy

        mask = torch.arange(waves.size(1), device=waves.device)[None] < wave_lengths[:, None]
        logits = self.model(input_values=waves, attention_mask=mask.long()).logits
        out_lengths = self.model._get_feat_extract_output_lengths(wave_lengths)
        return [self.tokenizer.decode(seq) for seq in ctc_greedy(torch.log_softmax(logits, -1), out_lengths)]


class WhisperFine(nn.Module):
    """Whisper encoder-decoder fine-tuned on (log-mel, text) pairs with its own tokenizer."""

    def __init__(self, name: str = "openai/whisper-tiny", language: str = "en", model=None, processor=None):
        super().__init__()
        if model is None or processor is None:
            from transformers import WhisperForConditionalGeneration, WhisperProcessor

            processor = processor or WhisperProcessor.from_pretrained(name)
            model = model or WhisperForConditionalGeneration.from_pretrained(name)
        self.model = model
        self.processor = processor
        self.language = language
        self.model.generation_config.language = language
        self.model.generation_config.task = "transcribe"
        self.model.generation_config.forced_decoder_ids = None

    def features(self, waves: list, sample_rate: int = 16_000) -> torch.Tensor:
        return self.processor.feature_extractor(waves, sampling_rate=sample_rate, return_tensors="pt").input_features

    def labels(self, texts: list[str]) -> torch.Tensor:
        batch = self.processor.tokenizer(texts, padding=True, return_tensors="pt")
        labels = batch.input_ids.masked_fill(batch.attention_mask == 0, -100)
        return labels

    def forward(self, input_features: torch.Tensor, labels: torch.Tensor):
        out = self.model(input_features=input_features, labels=labels)
        return {"loss": out.loss}

    @torch.no_grad()
    def transcribe(self, input_features: torch.Tensor, max_new_tokens: int = 128) -> list[str]:
        ids = self.model.generate(input_features=input_features, max_new_tokens=max_new_tokens)
        return [t.strip().lower() for t in self.processor.batch_decode(ids, skip_special_tokens=True)]


def char_vocab_for_hf(tokenizer) -> dict[str, int]:
    """A vocab.json-style mapping for Wav2Vec2CTCTokenizer from the repo's CharTokenizer
    ("|" stands for the space, as Hugging Face expects)."""
    vocab = {}
    for i, ch in enumerate(tokenizer.itos):
        if i < len(SPECIALS):
            vocab[["<pad>", "<s>", "</s>", "<unk>"][i]] = i
        else:
            vocab["|" if ch == " " else ch] = i
    return vocab

### 8.2 Fine-tune

Only runs when `MODEL` is `wav2vec2` or `whisper`. The data is loaded as 16 kHz waveforms
(wav2vec2) or Whisper log-mels, in duration-bucketed batches; the optimiser is AdamW with a
small learning rate and warm-up; validation WER is computed with greedy decoding every
`FT_EVAL_EVERY` steps and the best weights are saved to `OUTPUT_DIR`.

In [ ]:
FT_STEPS = 200 if SMOKE_TEST else 3000
FT_EVAL_EVERY = 100 if SMOKE_TEST else 500
FT_LR = 3e-5 if MODEL == "wav2vec2" else 1e-5
FT_BATCH = 4 if device.type == "cpu" else 8


def waveform_batches(utts, batch_size, shuffle, seed=0):
    sampler = BucketSampler([u.duration for u in utts], batch_size, 64, shuffle, seed)
    for idx in sampler:
        waves = [torch.as_tensor(load_audio(utts[i].audio, 16000)) for i in idx]
        lengths = torch.tensor([w.numel() for w in waves])
        padded = torch.zeros(len(waves), int(lengths.max()))
        for b, w in enumerate(waves):
            padded[b, : w.numel()] = w
        yield padded, lengths, [utts[i].text for i in idx]


if MODEL in ("wav2vec2", "whisper"):
    if MODEL == "wav2vec2":
        ft = Wav2Vec2Fine(tokenizer, PRETRAINED_NAME).to(device)
    else:
        ft = WhisperFine(PRETRAINED_NAME).to(device)
    optimizer = torch.optim.AdamW([p for p in ft.parameters() if p.requires_grad], lr=FT_LR, weight_decay=0.01)
    step, best_wer, t0, ft_log = 0, float("inf"), time.time(), []

    def ft_evaluate(max_batches=None):
        ft.eval()
        refs, hyps = [], []
        for i, (waves, lengths, texts) in enumerate(waveform_batches(val_utts, FT_BATCH, False)):
            if max_batches and i >= max_batches:
                break
            if MODEL == "wav2vec2":
                hyps += ft.transcribe(waves.to(device), lengths.to(device))
            else:
                hyps += ft.transcribe(ft.features([w[:n].numpy() for w, n in zip(waves, lengths)]).to(device))
            refs += texts
        ft.train()
        return error_report(refs, hyps), list(zip(refs[:3], hyps[:3]))

    ft.train()
    while step < FT_STEPS:
        for waves, lengths, texts in waveform_batches(train_utts, FT_BATCH, True, seed=step):
            lr = FT_LR * min(1.0, (step + 1) / 100)
            for g in optimizer.param_groups:
                g["lr"] = lr
            if MODEL == "wav2vec2":
                targets = [torch.tensor(tokenizer.encode(t)) for t in texts]
                tl = torch.tensor([t.numel() for t in targets])
                padded_t = torch.zeros(len(targets), int(tl.max()), dtype=torch.long)
                for b, t in enumerate(targets):
                    padded_t[b, : t.numel()] = t
                loss = ft(waves.to(device), lengths.to(device), padded_t.to(device), tl.to(device))["loss"]
            else:
                feats = ft.features([w[:n].numpy() for w, n in zip(waves, lengths)]).to(device)
                loss = ft(feats, ft.labels(texts).to(device))["loss"]
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ft.parameters(), 1.0)
            optimizer.step()
            step += 1
            if step % 20 == 0:
                print(f"step {step}  loss {float(loss):.3f}  lr {lr:.1e}  {time.time() - t0:.0f}s")
            if step % FT_EVAL_EVERY == 0 or step >= FT_STEPS:
                report, examples = ft_evaluate(max_batches=5 if SMOKE_TEST else None)
                ft_log.append((step, report["wer"], report["cer"]))
                print(f"  val WER {100 * report['wer']:.1f}%  CER {100 * report['cer']:.1f}%")
                for ref, hyp in examples:
                    print(f"    ref: {ref!r}\n    hyp: {hyp!r}")
                if report["wer"] < best_wer:
                    best_wer = report["wer"]
                    torch.save(ft.state_dict(), OUTPUT_DIR / "best_pretrained.pt")
                    (OUTPUT_DIR / "results.json").write_text(json.dumps({"model": MODEL, "name": PRETRAINED_NAME, "step": step,
                                                                          "wer": report["wer"], "cer": report["cer"]}))
            if step >= FT_STEPS:
                break
    print("best WER", best_wer)

## 9. Results

For the five built-in models: greedy vs beam search (and beam + a character 5-gram LM
trained on the training transcripts), the final WER / CER table, example transcripts and
the worst alignments.

In [ ]:
if MODEL in MODEL_TYPES:
    ckpt = load_checkpoint(OUTPUT_DIR / "best.pt", device)
    model, tokenizer = ckpt["model"], ckpt["tokenizer"]
    lm = NGramLM(order=5, vocab_size=tokenizer.vocab_size).train([tokenizer.encode(u.text) for u in train_utts])
    val_loader = make_loader(SpeechDataset(data_config, val_utts, tokenizer, train=False), batch_size, False)
    max_batches = 5 if SMOKE_TEST else None
    rows = []
    for name, kwargs in [("greedy", {}), ("beam 8", {"method": "beam", "beam": 8}),
                         ("beam 8 + 5-gram LM", {"method": "beam", "beam": 8, "lm": lm, "lm_weight": 0.3})]:
        if model.cfg.model == "rnnt" and "lm" in kwargs:
            continue
        res = evaluate(model, val_loader, tokenizer, device, max_batches=max_batches, **kwargs)
        rows.append((name, res["wer"], res["cer"]))
        print(f"{name:22} WER {100 * res['wer']:6.2f}%   CER {100 * res['cer']:6.2f}%")
    print("\nexamples (greedy):")
    res = evaluate(model, val_loader, tokenizer, device, max_batches=1)
    for ref, hyp in res["examples"]:
        print(f"  ref: {ref}\n  hyp: {hyp}\n")
    refs, hyps = [], []
    for batch in val_loader:
        ids = decode_batch(model, batch["feats"].to(device), batch["feat_lengths"].to(device))
        hyps += [tokenizer.decode(s) for s in ids]
        refs += batch["texts"]
        if max_batches and len(refs) >= max_batches * batch_size:
            break
    report = error_report(refs, hyps)
    worst = sorted(range(len(refs)), key=lambda i: -report["per_utterance_wer"][i])[:3]
    print("worst utterances:")
    for i in worst:
        print(format_alignment(refs[i], hyps[i]), "\n")
    plot_curves(OUTPUT_DIR / "log.csv")

## 10. Download the trained model

`best.pt` holds the weights, configs and tokenizer; use it with the repo:

```bash
stt-transcribe best.pt clip.wav --decode beam
```

On Kaggle, "Save Version" keeps `/kaggle/working`; on Colab with Drive the files are already
on Drive. Otherwise the zip below can be downloaded from the file browser.

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR.parent / f"stt-{MODEL}-{ENCODER}"), "zip", OUTPUT_DIR)
print("archive:", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")
if ON_COLAB and not USE_GOOGLE_DRIVE:
    from google.colab import files

    files.download(archive)